### **Installations, Imports, Configurations**

In [1]:
# ============================================================
# Cell 1 — Oracle-analysis dependencies
# ============================================================

import importlib.util
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version as pkg_version

REQUIRED_PACKAGES = {
    "pandas": "pandas",
    "numpy": "numpy",
    "sacrebleu": "sacrebleu",
    "sentencepiece": "sentencepiece",
    "tqdm": "tqdm",
    "packaging": "packaging",
}

missing_packages = [
    pip_spec
    for module_name, pip_spec in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]

print("Missing packages:", missing_packages)

if missing_packages:
    command = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-U",
        *missing_packages,
    ]

    print("Running:", " ".join(command))
    subprocess.check_call(command)
else:
    print("All oracle-analysis packages are installed.")

print("\nPython:", sys.version)
print("Executable:", sys.executable)

for package_name in REQUIRED_PACKAGES:
    try:
        print(f"{package_name}: {pkg_version(package_name)}")
    except PackageNotFoundError:
        print(f"{package_name}: version unavailable")

print("\nCell 1 completed.")

Missing packages: []
All oracle-analysis packages are installed.

Python: 3.11.15 (main, Jun 11 2026, 15:20:16) [GCC 14.3.0]
Executable: /home/mabdallah/alexandriax_mt_14d/envs/axmt_py311/bin/python
pandas: 3.0.3
numpy: 2.4.6
sacrebleu: 2.6.0
sentencepiece: 0.2.2
tqdm: 4.68.4
packaging: 26.2

Cell 1 completed.


In [2]:
# ============================================================
# Cell 2 — Imports, paths, candidates, and controls
# ============================================================

import hashlib
import json
import os
import random
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import sacrebleu

from IPython.display import display
from sacrebleu.metrics import BLEU, CHRF
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 3407
random.seed(SEED)
np.random.seed(SEED)

PROJECT_DIR = Path(
    os.environ.get(
        "AXMT_HOME",
        str(Path.home() / "alexandriax_mt_14d"),
    )
).expanduser()

INFERENCE_VARIANTS_ROOT = Path(
    os.environ.get(
        "AXMT_VARIANTS_ROOT",
        str(PROJECT_DIR / "inference_variants"),
    )
).expanduser()

ORACLE_DIR = (
    INFERENCE_VARIANTS_ROOT
    / "_oracle_analysis_v1"
)
ORACLE_DIR.mkdir(parents=True, exist_ok=True)

CANDIDATE_VARIANTS = [
    "00_previous_official_control",
    "01_exact_training_parity",
    "02_metadata_no_shots",
    "03_retrieved_two_shot",
    "04_training_parity_with_participants",
    "05_retrieved_two_shot_with_participants",
    "06_ckpt16500_retrieved_two_shot",
    "07_ckpt16000_retrieved_two_shot",
]

CURRENT_SYSTEM_VARIANT = (
    "92_mixed_best_checkpoint_variant_per_country"
)

CHECKPOINT_BY_VARIANT = {
    name: (
        "16500"
        if name.startswith("06_")
        else "16000"
        if name.startswith("07_")
        else "16600"
    )
    for name in CANDIDATE_VARIANTS
}

EXPECTED_DEV_TURNS = 12_250
EXPECTED_COUNTRIES = 11

EXPECTED_CURRENT_SPBLEU = 30.928003
EXPECTED_CURRENT_CHRFPP = 45.594526
CURRENT_SCORE_TOLERANCE = 0.05

OOF_FOLDS = 5
OOF_MIN_FINE_GROUP_ROWS = 25

RUN_COORDINATE_ORACLE = True
COORDINATE_MAX_PASSES = 4
COORDINATE_TOLERANCE = 1e-12

BASE_COLUMNS = [
    "source_id",
    "config",
    "conversation_id",
    "turn_order",
    "source_text",
    "reference_arabic",
]

PREDICTION_COLUMNS = [
    f"prediction__{name}"
    for name in CANDIDATE_VARIANTS
]

VARIANT_TO_INDEX = {
    name: index
    for index, name in enumerate(CANDIDATE_VARIANTS)
}

if not INFERENCE_VARIANTS_ROOT.exists():
    raise FileNotFoundError(
        f"Inference variants root not found: "
        f"{INFERENCE_VARIANTS_ROOT}"
    )

missing_folders = [
    name
    for name in CANDIDATE_VARIANTS
    if not (
        INFERENCE_VARIANTS_ROOT / name
    ).exists()
]

if missing_folders:
    raise FileNotFoundError(
        "Missing candidate folders:\n - "
        + "\n - ".join(missing_folders)
    )

print("Project:", PROJECT_DIR)
print("Variants root:", INFERENCE_VARIANTS_ROOT)
print("Oracle output:", ORACLE_DIR)
print("Candidates:", len(CANDIDATE_VARIANTS))
print("SacreBLEU:", sacrebleu.__version__)
print("Coordinate oracle:", RUN_COORDINATE_ORACLE)

Project: /home/mabdallah/alexandriax_mt_14d
Variants root: /home/mabdallah/alexandriax_mt_14d/inference_variants
Oracle output: /home/mabdallah/alexandriax_mt_14d/inference_variants/_oracle_analysis_v1
Candidates: 8
SacreBLEU: 2.6.0
Coordinate oracle: True


### **Load & Align Saved Predictions**

In [3]:
# ============================================================
# Cell 3 — Load and align saved predictions
#
# Reads:
#   variants 00–07
#   current country mixture 92
#   optional official DEV metadata cache
# ============================================================

def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with open(path, "rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def read_prediction_csv(path):
    frame = pd.read_csv(
        path,
        dtype={"source_id": "string"},
    )

    if "source_id" not in frame.columns:
        raise ValueError(
            f"source_id missing from {path}"
        )

    if "prediction" not in frame.columns:
        raise ValueError(
            f"prediction missing from {path}"
        )

    if (
        "config" not in frame.columns
        and "country" in frame.columns
    ):
        frame["config"] = frame["country"]

    if "reference_arabic" not in frame.columns:
        for fallback_column in [
            "target_arabic",
            "reference",
        ]:
            if fallback_column in frame.columns:
                frame["reference_arabic"] = (
                    frame[fallback_column]
                )
                break

    frame["source_id"] = (
        frame["source_id"]
        .astype(str)
    )

    frame["prediction"] = (
        frame["prediction"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    if frame["source_id"].duplicated().any():
        raise ValueError(
            f"Duplicate source_id values in {path}"
        )

    empty_predictions = int(
        frame["prediction"].eq("").sum()
    )

    if empty_predictions:
        raise ValueError(
            f"{empty_predictions} empty predictions "
            f"in {path}"
        )

    return frame


candidate_frames = {}
candidate_paths = {}
candidate_hashes = {}

for variant_name in CANDIDATE_VARIANTS:
    variant_dir = (
        INFERENCE_VARIANTS_ROOT
        / variant_name
    )

    scored_path = (
        variant_dir
        / "scored_turn_predictions.csv"
    )

    raw_path = (
        variant_dir
        / "turn_predictions.csv"
    )

    prediction_path = (
        scored_path
        if scored_path.exists()
        else raw_path
    )

    if not prediction_path.exists():
        raise FileNotFoundError(
            f"No saved predictions for {variant_name}"
        )

    frame = read_prediction_csv(
        prediction_path
    )

    candidate_frames[variant_name] = frame
    candidate_paths[variant_name] = (
        prediction_path
    )
    candidate_hashes[variant_name] = (
        file_sha256(prediction_path)
    )

anchor_name = CANDIDATE_VARIANTS[0]
anchor_df = candidate_frames[
    anchor_name
].copy()

missing_anchor_columns = (
    set(BASE_COLUMNS)
    - set(anchor_df.columns)
)

if missing_anchor_columns:
    raise ValueError(
        f"{anchor_name} is missing columns: "
        f"{sorted(missing_anchor_columns)}\n"
        "Use scored_turn_predictions.csv."
    )

anchor_df["config"] = (
    anchor_df["config"]
    .astype(str)
)

anchor_df["conversation_id"] = (
    anchor_df["conversation_id"]
    .astype(str)
)

anchor_df["turn_order"] = (
    pd.to_numeric(
        anchor_df["turn_order"],
        errors="raise",
    )
    .astype(int)
)

anchor_df["source_text"] = (
    anchor_df["source_text"]
    .fillna("")
    .astype(str)
)

anchor_df["reference_arabic"] = (
    anchor_df["reference_arabic"]
    .fillna("")
    .astype(str)
    .str.strip()
)

if anchor_df["reference_arabic"].eq("").any():
    raise ValueError(
        "Empty official references detected."
    )

anchor_df = (
    anchor_df
    .sort_values(
        [
            "config",
            "conversation_id",
            "turn_order",
            "source_id",
        ]
    )
    .reset_index(drop=True)
)

if len(anchor_df) != EXPECTED_DEV_TURNS:
    raise ValueError(
        f"Expected {EXPECTED_DEV_TURNS} turns, "
        f"found {len(anchor_df)}"
    )

if (
    anchor_df["config"].nunique()
    != EXPECTED_COUNTRIES
):
    raise ValueError(
        f"Expected {EXPECTED_COUNTRIES} countries, "
        f"found {anchor_df['config'].nunique()}"
    )

expected_ids = set(
    anchor_df["source_id"]
)

analysis_df = anchor_df[
    BASE_COLUMNS
].copy()

for variant_name in CANDIDATE_VARIANTS:
    frame = candidate_frames[
        variant_name
    ].copy()

    actual_ids = set(
        frame["source_id"]
    )

    if actual_ids != expected_ids:
        raise ValueError(
            f"ID mismatch for {variant_name}: "
            f"missing={len(expected_ids - actual_ids)}, "
            f"extra={len(actual_ids - expected_ids)}"
        )

    if "config" in frame.columns:
        check_df = anchor_df[
            ["source_id", "config"]
        ].merge(
            frame[
                ["source_id", "config"]
            ],
            on="source_id",
            suffixes=(
                "__anchor",
                "__candidate",
            ),
            validate="one_to_one",
        )

        if not (
            check_df["config__anchor"]
            .astype(str)
            .equals(
                check_df["config__candidate"]
                .astype(str)
            )
        ):
            raise ValueError(
                f"Country mismatch for {variant_name}"
            )

    if "reference_arabic" in frame.columns:
        check_df = anchor_df[
            ["source_id", "reference_arabic"]
        ].merge(
            frame[
                [
                    "source_id",
                    "reference_arabic",
                ]
            ],
            on="source_id",
            suffixes=(
                "__anchor",
                "__candidate",
            ),
            validate="one_to_one",
        )

        left_references = (
            check_df[
                "reference_arabic__anchor"
            ]
            .fillna("")
            .astype(str)
        )

        right_references = (
            check_df[
                "reference_arabic__candidate"
            ]
            .fillna("")
            .astype(str)
        )

        if not left_references.equals(
            right_references
        ):
            raise ValueError(
                f"Reference mismatch for {variant_name}"
            )

    prediction_column = (
        f"prediction__{variant_name}"
    )

    analysis_df = analysis_df.merge(
        frame[
            ["source_id", "prediction"]
        ].rename(
            columns={
                "prediction": prediction_column
            }
        ),
        on="source_id",
        how="left",
        validate="one_to_one",
    )

# ------------------------------------------------------------
# Add optional metadata from the prepared DEV cache
# ------------------------------------------------------------

metadata_cache_path = (
    INFERENCE_VARIANTS_ROOT
    / "_shared_cache"
    / "paired_dev_train_v3"
    / "official_dev_df.pkl"
)

optional_metadata_columns = [
    "dialect",
    "domain",
    "participants",
    "speaker",
    "gender_direction",
    "previous_english_turns",
    "translator_id",
    "reviewer_id",
]

loaded_metadata_columns = []

if metadata_cache_path.exists():
    metadata_df = pd.read_pickle(
        metadata_cache_path
    ).copy()

    metadata_df["source_id"] = (
        metadata_df["source_id"]
        .astype(str)
    )

    loaded_metadata_columns = [
        column
        for column in optional_metadata_columns
        if column in metadata_df.columns
    ]

    analysis_df = analysis_df.merge(
        metadata_df[
            [
                "source_id",
                *loaded_metadata_columns,
            ]
        ],
        on="source_id",
        how="left",
        validate="one_to_one",
    )

    print(
        "Loaded metadata:",
        loaded_metadata_columns,
    )
else:
    print(
        "Metadata cache not found. "
        "Domain-aware analyses will be skipped."
    )

analysis_df["turn_bucket"] = pd.cut(
    analysis_df["turn_order"],
    bins=[
        -np.inf,
        1,
        2,
        3,
        5,
        np.inf,
    ],
    labels=[
        "1",
        "2",
        "3",
        "4-5",
        "6+",
    ],
).astype(str)

for column in [
    "domain",
    "dialect",
    "speaker",
    "gender_direction",
]:
    if column in analysis_df.columns:
        analysis_df[column] = (
            analysis_df[column]
            .fillna("unknown")
            .astype(str)
        )

# ------------------------------------------------------------
# Load current mixed system 92
# ------------------------------------------------------------

current_dir = (
    INFERENCE_VARIANTS_ROOT
    / CURRENT_SYSTEM_VARIANT
)

current_path = (
    current_dir
    / "turn_predictions.csv"
)

if not current_path.exists():
    current_path = (
        current_dir
        / "scored_turn_predictions.csv"
    )

if not current_path.exists():
    raise FileNotFoundError(
        f"Current system predictions missing: "
        f"{current_dir}"
    )

current_raw_df = read_prediction_csv(
    current_path
)

if (
    set(current_raw_df["source_id"])
    != expected_ids
):
    raise ValueError(
        "Current system IDs do not match DEV."
    )

current_columns = [
    "source_id",
    "prediction",
]

for column in [
    "selected_from_variant",
    "selected_from_checkpoint",
]:
    if column in current_raw_df.columns:
        current_columns.append(column)

current_df = anchor_df[
    BASE_COLUMNS
].merge(
    current_raw_df[current_columns],
    on="source_id",
    how="left",
    validate="one_to_one",
).rename(
    columns={
        "prediction": "current_prediction"
    }
)

if (
    "selected_from_variant"
    not in current_df.columns
):
    current_df[
        "selected_from_variant"
    ] = pd.NA

current_df.loc[
    ~current_df[
        "selected_from_variant"
    ].isin(CANDIDATE_VARIANTS),
    "selected_from_variant",
] = pd.NA

candidate_lookup_df = (
    analysis_df
    .set_index("source_id")
)

# Remove stale labels.
for variant_name in CANDIDATE_VARIANTS:
    candidate_values = (
        current_df["source_id"]
        .map(
            candidate_lookup_df[
                f"prediction__{variant_name}"
            ]
        )
    )

    stale_mask = (
        current_df[
            "selected_from_variant"
        ].eq(variant_name)
        & current_df[
            "current_prediction"
        ].ne(candidate_values)
    )

    current_df.loc[
        stale_mask,
        "selected_from_variant",
    ] = pd.NA

# Recover missing labels from exact prediction matches.
unmatched_mask = current_df[
    "selected_from_variant"
].isna()

for variant_name in CANDIDATE_VARIANTS:
    candidate_values = (
        current_df["source_id"]
        .map(
            candidate_lookup_df[
                f"prediction__{variant_name}"
            ]
        )
    )

    match_mask = (
        unmatched_mask
        & current_df[
            "current_prediction"
        ].eq(candidate_values)
    )

    current_df.loc[
        match_mask,
        "selected_from_variant",
    ] = variant_name

    unmatched_mask = current_df[
        "selected_from_variant"
    ].isna()

if unmatched_mask.any():
    raise ValueError(
        f"{int(unmatched_mask.sum())} current "
        "predictions do not match variants 00–07."
    )

current_df[
    "selected_from_checkpoint"
] = current_df[
    "selected_from_variant"
].map(CHECKPOINT_BY_VARIANT)

analysis_df = analysis_df.merge(
    current_df[
        [
            "source_id",
            "current_prediction",
            "selected_from_variant",
            "selected_from_checkpoint",
        ]
    ],
    on="source_id",
    how="left",
    validate="one_to_one",
)

analysis_df.to_pickle(
    ORACLE_DIR
    / "aligned_analysis_frame.pkl"
)

input_inventory_df = pd.DataFrame([
    {
        "variant": variant_name,
        "checkpoint": (
            CHECKPOINT_BY_VARIANT[
                variant_name
            ]
        ),
        "rows": len(
            candidate_frames[
                variant_name
            ]
        ),
        "prediction_path": str(
            candidate_paths[
                variant_name
            ]
        ),
        "sha256": (
            candidate_hashes[
                variant_name
            ]
        ),
    }
    for variant_name in CANDIDATE_VARIANTS
])

input_inventory_df.to_csv(
    ORACLE_DIR
    / "input_inventory.csv",
    index=False,
    encoding="utf-8-sig",
)

print("\nAlignment completed.")
print("Turns:", len(analysis_df))
print(
    "Countries:",
    analysis_df["config"].nunique(),
)
print(
    "Candidate prediction columns:",
    len(PREDICTION_COLUMNS),
)

print("\nCurrent mixture expert usage:")
display(
    analysis_df[
        "selected_from_variant"
    ]
    .value_counts()
    .rename_axis("variant")
    .reset_index(name="turns")
)

Loaded metadata: ['dialect', 'domain', 'participants', 'speaker', 'gender_direction', 'previous_english_turns', 'translator_id', 'reviewer_id']

Alignment completed.
Turns: 12250
Countries: 11
Candidate prediction columns: 8

Current mixture expert usage:


,variant,turns
0,05_retrieved_two_shot_with_participants,4451
1,07_ckpt16000_retrieved_two_shot,3348
2,06_ckpt16500_retrieved_two_shot,3338
3,03_retrieved_two_shot,1113


### **Reproduce official scores**

In [4]:
# ============================================================
# Cell 4 — Reproduce official spBLEU and chrF++ scores
# ============================================================

def official_macro_scores(
    frame,
    prediction_column,
    system_name,
):
    country_rows = []

    countries = sorted(
        frame["config"]
        .astype(str)
        .unique()
    )

    for country in countries:
        country_df = frame.loc[
            frame["config"]
            .astype(str)
            .eq(country)
        ]

        predictions = (
            country_df[prediction_column]
            .astype(str)
            .tolist()
        )

        references = (
            country_df[
                "reference_arabic"
            ]
            .astype(str)
            .tolist()
        )

        country_rows.append({
            "system": system_name,
            "country": country,
            "turns": len(country_df),
            "spBLEU": float(
                sacrebleu.corpus_bleu(
                    predictions,
                    [references],
                    tokenize="flores200",
                ).score
            ),
            "chrF++": float(
                sacrebleu.corpus_chrf(
                    predictions,
                    [references],
                    word_order=2,
                ).score
            ),
        })

    country_scores = pd.DataFrame(
        country_rows
    )

    summary = {
        "system": system_name,
        "macro_spBLEU": float(
            country_scores[
                "spBLEU"
            ].mean()
        ),
        "macro_chrFpp": float(
            country_scores[
                "chrF++"
            ].mean()
        ),
        "countries": len(country_scores),
        "turns": int(
            country_scores[
                "turns"
            ].sum()
        ),
    }

    return summary, country_scores


candidate_summary_rows = []
candidate_country_parts = []

for variant_name in CANDIDATE_VARIANTS:
    summary, country_scores = (
        official_macro_scores(
            analysis_df,
            f"prediction__{variant_name}",
            variant_name,
        )
    )

    summary["checkpoint"] = (
        CHECKPOINT_BY_VARIANT[
            variant_name
        ]
    )

    candidate_summary_rows.append(
        summary
    )
    candidate_country_parts.append(
        country_scores
    )

current_summary, current_country_scores = (
    official_macro_scores(
        analysis_df,
        "current_prediction",
        CURRENT_SYSTEM_VARIANT,
    )
)

candidate_summary_df = pd.DataFrame(
    candidate_summary_rows
)

candidate_country_scores_df = pd.concat(
    candidate_country_parts,
    ignore_index=True,
)

candidate_ranking_df = (
    candidate_summary_df
    .sort_values(
        [
            "macro_spBLEU",
            "macro_chrFpp",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)

candidate_ranking_df.insert(
    0,
    "rank",
    np.arange(
        1,
        len(candidate_ranking_df) + 1,
    ),
)

candidate_ranking_df[
    "delta_vs_current_spBLEU"
] = (
    candidate_ranking_df[
        "macro_spBLEU"
    ]
    - current_summary[
        "macro_spBLEU"
    ]
)

candidate_ranking_df.to_csv(
    ORACLE_DIR
    / "candidate_official_scores.csv",
    index=False,
    encoding="utf-8-sig",
)

candidate_country_scores_df.to_csv(
    ORACLE_DIR
    / "candidate_per_country_scores.csv",
    index=False,
    encoding="utf-8-sig",
)

current_country_scores.to_csv(
    ORACLE_DIR
    / "current_system_per_country_scores.csv",
    index=False,
    encoding="utf-8-sig",
)

print(
    f"Current 92 system: "
    f"{current_summary['macro_spBLEU']:.6f} "
    f"spBLEU / "
    f"{current_summary['macro_chrFpp']:.6f} "
    f"chrF++"
)

current_spbleu_difference = abs(
    current_summary["macro_spBLEU"]
    - EXPECTED_CURRENT_SPBLEU
)

current_chrf_difference = abs(
    current_summary["macro_chrFpp"]
    - EXPECTED_CURRENT_CHRFPP
)

if (
    current_spbleu_difference
    > CURRENT_SCORE_TOLERANCE
    or current_chrf_difference
    > CURRENT_SCORE_TOLERANCE
):
    raise RuntimeError(
        "\nThe current 92 mixture does not match "
        "the known final system.\n"
        "Expected approximately "
        "30.928003 spBLEU / "
        "45.594526 chrF++.\n"
        "The 92 folder may have been overwritten. "
        "Restore it before continuing."
    )

print("\nOfficial score validation passed.")

display(
    candidate_ranking_df[
        [
            "rank",
            "system",
            "checkpoint",
            "macro_spBLEU",
            "macro_chrFpp",
            "delta_vs_current_spBLEU",
        ]
    ]
)

country_best_rows = (
    candidate_country_scores_df
    .sort_values(
        [
            "country",
            "spBLEU",
            "chrF++",
            "system",
        ],
        ascending=[
            True,
            False,
            False,
            True,
        ],
    )
    .groupby(
        "country",
        as_index=False,
    )
    .first()
)

COUNTRY_BEST_VARIANT = dict(
    zip(
        country_best_rows["country"],
        country_best_rows["system"],
    )
)

print("\nBest candidate per country:")
display(
    country_best_rows[
        [
            "country",
            "system",
            "spBLEU",
            "chrF++",
        ]
    ]
)

Current 92 system: 30.928003 spBLEU / 45.594526 chrF++

Official score validation passed.


,rank,system,checkpoint,macro_spBLEU,macro_chrFpp,delta_vs_current_spBLEU
0,1,05_retrieved_two_shot_with_participants,16600,30.695985,45.484384,-0.232018
1,2,03_retrieved_two_shot,16600,30.693517,45.491812,-0.234486
2,3,07_ckpt16000_retrieved_two_shot,16000,30.652061,45.303998,-0.275942
3,4,01_exact_training_parity,16600,30.565838,45.381164,-0.362165
4,5,02_metadata_no_shots,16600,30.555001,45.341404,-0.373002
5,6,04_training_parity_with_participants,16600,30.554500,45.409125,-0.373502
6,7,06_ckpt16500_retrieved_two_shot,16500,30.550917,45.519670,-0.377085
7,8,00_previous_official_control,16600,30.165227,45.079987,-0.762775



Best candidate per country:


,country,system,spBLEU,chrF++
0,EG,05_retrieved_two_shot_with_participants,32.903189,46.892198
1,JO,03_retrieved_two_shot,35.301943,49.377895
2,LB,07_ckpt16000_retrieved_two_shot,31.701457,45.871215
3,MA,05_retrieved_two_shot_with_participants,23.530143,39.376601
4,MR,07_ckpt16000_retrieved_two_shot,17.380438,33.733013
5,OM,05_retrieved_two_shot_with_participants,36.147408,49.843410
6,PS,06_ckpt16500_retrieved_two_shot,33.418402,47.762506
7,SA,06_ckpt16500_retrieved_two_shot,33.166094,48.135685
8,SY,05_retrieved_two_shot_with_participants,39.993111,53.985574
9,TN,07_ckpt16000_retrieved_two_shot,29.285225,43.452865


### **Candidate diversity and disagreement**

In [5]:
# ============================================================
# Cell 5 — Candidate diversity and disagreement
# ============================================================

candidate_matrix = (
    analysis_df[
        PREDICTION_COLUMNS
    ]
    .astype(str)
    .to_numpy()
)

analysis_df[
    "unique_candidate_outputs"
] = (
    pd.DataFrame(candidate_matrix)
    .nunique(axis=1)
    .to_numpy()
)

analysis_df[
    "all_candidates_identical"
] = analysis_df[
    "unique_candidate_outputs"
].eq(1)

pair_rows = []

pairwise_agreement_df = pd.DataFrame(
    np.eye(
        len(CANDIDATE_VARIANTS),
        dtype=float,
    ),
    index=CANDIDATE_VARIANTS,
    columns=CANDIDATE_VARIANTS,
)

for left_index, left_name in enumerate(
    CANDIDATE_VARIANTS
):
    for right_index in range(
        left_index + 1,
        len(CANDIDATE_VARIANTS),
    ):
        right_name = (
            CANDIDATE_VARIANTS[
                right_index
            ]
        )

        agreement = float(
            np.mean(
                candidate_matrix[
                    :,
                    left_index,
                ]
                == candidate_matrix[
                    :,
                    right_index,
                ]
            )
        )

        pairwise_agreement_df.loc[
            left_name,
            right_name,
        ] = agreement

        pairwise_agreement_df.loc[
            right_name,
            left_name,
        ] = agreement

        pair_rows.append({
            "left_variant": left_name,
            "right_variant": right_name,
            "exact_agreement_rate": agreement,
            "disagreement_rate": (
                1.0 - agreement
            ),
        })

pairwise_long_df = (
    pd.DataFrame(pair_rows)
    .sort_values(
        "exact_agreement_rate"
    )
    .reset_index(drop=True)
)

diversity_by_country_df = (
    analysis_df
    .groupby(
        "config",
        as_index=False,
    )
    .agg(
        turns=("source_id", "size"),
        mean_unique_outputs=(
            "unique_candidate_outputs",
            "mean",
        ),
        all_same_rate=(
            "all_candidates_identical",
            "mean",
        ),
    )
    .rename(
        columns={"config": "country"}
    )
)

diversity_by_country_df[
    "any_disagreement_rate"
] = (
    1.0
    - diversity_by_country_df[
        "all_same_rate"
    ]
)

pairwise_agreement_df.to_csv(
    ORACLE_DIR
    / "pairwise_exact_agreement_matrix.csv",
    encoding="utf-8-sig",
)

pairwise_long_df.to_csv(
    ORACLE_DIR
    / "pairwise_exact_agreement_long.csv",
    index=False,
    encoding="utf-8-sig",
)

diversity_by_country_df.to_csv(
    ORACLE_DIR
    / "candidate_diversity_by_country.csv",
    index=False,
    encoding="utf-8-sig",
)

any_disagreement_rate = (
    1.0
    - float(
        analysis_df[
            "all_candidates_identical"
        ].mean()
    )
)

three_or_more_rate = float(
    analysis_df[
        "unique_candidate_outputs"
    ].ge(3).mean()
)

print(
    "Turns with at least two outputs:",
    f"{any_disagreement_rate:.2%}",
)

print(
    "Turns with at least three outputs:",
    f"{three_or_more_rate:.2%}",
)

print(
    "Pairwise agreement range:",
    f"{pairwise_long_df['exact_agreement_rate'].min():.2%}",
    "to",
    f"{pairwise_long_df['exact_agreement_rate'].max():.2%}",
)

print("\nMost different candidate pairs:")
display(pairwise_long_df.head(10))

print("\nCandidate diversity by country:")
display(
    diversity_by_country_df.sort_values(
        "mean_unique_outputs",
        ascending=False,
    )
)

Turns with at least two outputs: 94.36%
Turns with at least three outputs: 80.26%
Pairwise agreement range: 12.89% to 53.64%

Most different candidate pairs:


,left_variant,right_variant,exact_agreement_rate,disagreement_rate
0,00_previous_official_control,07_ckpt16000_retrieved_two_shot,0.128898,0.871102
1,02_metadata_no_shots,07_ckpt16000_retrieved_two_shot,0.145714,0.854286
2,04_training_parity_with_participants,07_ckpt16000_retrieved_two_shot,0.150612,0.849388
3,01_exact_training_parity,07_ckpt16000_retrieved_two_shot,0.151510,0.848490
4,05_retrieved_two_shot_with_participants,07_ckpt16000_retrieved_two_shot,0.162531,0.837469
5,03_retrieved_two_shot,07_ckpt16000_retrieved_two_shot,0.171755,0.828245
6,06_ckpt16500_retrieved_two_shot,07_ckpt16000_retrieved_two_shot,0.174367,0.825633
7,00_previous_official_control,06_ckpt16500_retrieved_two_shot,0.211918,0.788082
8,02_metadata_no_shots,06_ckpt16500_retrieved_two_shot,0.247510,0.752490
9,01_exact_training_parity,06_ckpt16500_retrieved_two_shot,0.257714,0.742286



Candidate diversity by country:


,country,turns,mean_unique_outputs,all_same_rate,any_disagreement_rate
4,MR,1114,5.333034,0.017056,0.982944
3,MA,1110,5.156757,0.032432,0.967568
10,YE,1118,4.614490,0.041145,0.958855
9,TN,1116,4.336022,0.058244,0.941756
2,LB,1118,4.122540,0.069767,0.930233
5,OM,1109,4.049594,0.063120,0.936880
0,EG,1113,4.048518,0.066487,0.933513
6,PS,1110,4.021622,0.063964,0.936036
7,SA,1110,3.990090,0.059459,0.940541
1,JO,1113,3.932615,0.081761,0.918239


### **Sentence-level metric cache**

In [6]:
# ============================================================
# Cell 6 — Sentence-level diagnostic scores
#
# sentence_spBLEU uses effective_order=True.
# It is a diagnostic metric, not the official corpus metric.
# ============================================================

sentence_cache_path = (
    ORACLE_DIR
    / "sentence_candidate_scores.pkl"
)

sentence_manifest_path = (
    ORACLE_DIR
    / "sentence_candidate_scores_manifest.json"
)

sentence_cache_payload = {
    "candidate_hashes": candidate_hashes,
    "rows": len(analysis_df),
    "sacrebleu_version": (
        sacrebleu.__version__
    ),
    "sentence_bleu": {
        "tokenize": "flores200",
        "smooth_method": "exp",
        "effective_order": True,
    },
    "sentence_chrf": {
        "word_order": 2,
    },
}

sentence_cache_fingerprint = (
    hashlib.sha256(
        json.dumps(
            sentence_cache_payload,
            ensure_ascii=False,
            sort_keys=True,
        ).encode("utf-8")
    ).hexdigest()
)

use_sentence_cache = False

if (
    sentence_cache_path.exists()
    and sentence_manifest_path.exists()
):
    with open(
        sentence_manifest_path,
        "r",
        encoding="utf-8",
    ) as handle:
        existing_manifest = json.load(
            handle
        )

    use_sentence_cache = (
        existing_manifest.get(
            "fingerprint"
        )
        == sentence_cache_fingerprint
    )

if use_sentence_cache:
    sentence_scores_df = (
        pd.read_pickle(
            sentence_cache_path
        )
    )

    print(
        "Loaded sentence cache:",
        sentence_cache_path,
    )

else:
    sentence_bleu_metric = BLEU(
        tokenize="flores200",
        smooth_method="exp",
        effective_order=True,
    )

    sentence_chrf_metric = CHRF(
        word_order=2
    )

    references = (
        analysis_df[
            "reference_arabic"
        ]
        .astype(str)
        .tolist()
    )

    sentence_parts = []

    for variant_name in CANDIDATE_VARIANTS:
        predictions = (
            analysis_df[
                f"prediction__{variant_name}"
            ]
            .astype(str)
            .tolist()
        )

        # Efficient BLEU tokenization and statistics.
        bleu_statistics = (
            sentence_bleu_metric
            ._extract_corpus_statistics(
                predictions,
                [references],
            )
        )

        sentence_bleu_scores = [
            float(
                sentence_bleu_metric
                ._compute_score_from_stats(
                    statistics
                )
                .score
            )
            for statistics in bleu_statistics
        ]

        sentence_chrf_scores = [
            float(
                sentence_chrf_metric
                .sentence_score(
                    prediction,
                    [reference],
                )
                .score
            )
            for prediction, reference in tqdm(
                zip(
                    predictions,
                    references,
                ),
                total=len(references),
                desc=(
                    "Sentence chrF++ — "
                    f"{variant_name[:28]}"
                ),
            )
        ]

        sentence_parts.append(
            pd.DataFrame({
                "source_id": (
                    analysis_df[
                        "source_id"
                    ].to_numpy()
                ),
                "config": (
                    analysis_df[
                        "config"
                    ].to_numpy()
                ),
                "variant": variant_name,
                "checkpoint": (
                    CHECKPOINT_BY_VARIANT[
                        variant_name
                    ]
                ),
                "sentence_spBLEU": (
                    sentence_bleu_scores
                ),
                "sentence_chrFpp": (
                    sentence_chrf_scores
                ),
            })
        )

    sentence_scores_df = pd.concat(
        sentence_parts,
        ignore_index=True,
    )

    sentence_scores_df.to_pickle(
        sentence_cache_path
    )

    with open(
        sentence_manifest_path,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            {
                **sentence_cache_payload,
                "fingerprint": (
                    sentence_cache_fingerprint
                ),
                "created_at": time.strftime(
                    "%Y-%m-%d %H:%M:%S"
                ),
            },
            handle,
            ensure_ascii=False,
            indent=2,
        )

    print(
        "Saved sentence cache:",
        sentence_cache_path,
    )

expected_sentence_rows = (
    len(analysis_df)
    * len(CANDIDATE_VARIANTS)
)

if (
    len(sentence_scores_df)
    != expected_sentence_rows
):
    raise ValueError(
        "Sentence-cache row mismatch: "
        f"expected {expected_sentence_rows}, "
        f"found {len(sentence_scores_df)}"
    )

if sentence_scores_df[
    ["source_id", "variant"]
].duplicated().any():
    raise ValueError(
        "Duplicate source_id × variant "
        "in sentence cache."
    )

print(
    "Sentence score rows:",
    len(sentence_scores_df),
)

print(
    "Cache fingerprint:",
    sentence_cache_fingerprint[:20],
)

Loaded sentence cache: /home/mabdallah/alexandriax_mt_14d/inference_variants/_oracle_analysis_v1/sentence_candidate_scores.pkl
Sentence score rows: 98000
Cache fingerprint: 650167bcaf3008a5718a


### **Per-turn oracle ceilings**

In [7]:
# ============================================================
# Cell 7 — Per-turn reference oracles
#
# 1. Stable oracle: sentence chrF++, BLEU tie-break
# 2. BLEU oracle: sentence BLEU, chrF++ tie-break
#
# These use DEV references and must never be submitted.
# ============================================================

def build_sentence_oracle(
    primary_metric,
    secondary_metric,
    oracle_name,
):
    winners_df = (
        sentence_scores_df
        .sort_values(
            [
                "source_id",
                primary_metric,
                secondary_metric,
                "variant",
            ],
            ascending=[
                True,
                False,
                False,
                True,
            ],
        )
        .groupby(
            "source_id",
            as_index=False,
            sort=False,
        )
        .first()
        [
            [
                "source_id",
                "variant",
                "checkpoint",
                "sentence_spBLEU",
                "sentence_chrFpp",
            ]
        ]
    )

    oracle_df = (
        analysis_df[
            BASE_COLUMNS
        ]
        .merge(
            winners_df,
            on="source_id",
            how="left",
            validate="one_to_one",
        )
        .rename(
            columns={
                "variant": (
                    "selected_variant"
                ),
                "checkpoint": (
                    "selected_checkpoint"
                ),
            }
        )
    )

    prediction_lookup = (
        analysis_df
        .set_index("source_id")
    )

    oracle_df["prediction"] = [
        prediction_lookup.at[
            source_id,
            f"prediction__{variant_name}",
        ]
        for source_id, variant_name in zip(
            oracle_df["source_id"],
            oracle_df[
                "selected_variant"
            ],
        )
    ]

    summary, country_scores = (
        official_macro_scores(
            oracle_df,
            "prediction",
            oracle_name,
        )
    )

    oracle_df.to_csv(
        ORACLE_DIR
        / (
            f"{oracle_name}_"
            "diagnostic_predictions.csv"
        ),
        index=False,
        encoding="utf-8-sig",
    )

    return (
        oracle_df,
        summary,
        country_scores,
    )


(
    sentence_chrf_oracle_df,
    sentence_chrf_oracle_summary,
    sentence_chrf_country_scores,
) = build_sentence_oracle(
    primary_metric="sentence_chrFpp",
    secondary_metric="sentence_spBLEU",
    oracle_name="oracle_sentence_chrfpp",
)

(
    sentence_bleu_oracle_df,
    sentence_bleu_oracle_summary,
    sentence_bleu_country_scores,
) = build_sentence_oracle(
    primary_metric="sentence_spBLEU",
    secondary_metric="sentence_chrFpp",
    oracle_name="oracle_sentence_spbleu",
)

sentence_oracle_summary_df = pd.DataFrame([
    sentence_chrf_oracle_summary,
    sentence_bleu_oracle_summary,
])

sentence_oracle_summary_df[
    "delta_vs_current_spBLEU"
] = (
    sentence_oracle_summary_df[
        "macro_spBLEU"
    ]
    - current_summary[
        "macro_spBLEU"
    ]
)

sentence_oracle_summary_df[
    "delta_vs_current_chrFpp"
] = (
    sentence_oracle_summary_df[
        "macro_chrFpp"
    ]
    - current_summary[
        "macro_chrFpp"
    ]
)

sentence_oracle_summary_df.to_csv(
    ORACLE_DIR
    / "sentence_oracle_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

winner_counts_df = (
    sentence_chrf_oracle_df
    .groupby(
        [
            "config",
            "selected_variant",
        ],
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "config": "country",
            "size": "wins",
        }
    )
)

winner_counts_df[
    "country_win_rate"
] = (
    winner_counts_df["wins"]
    / winner_counts_df
    .groupby("country")["wins"]
    .transform("sum")
)

winner_counts_df.to_csv(
    ORACLE_DIR
    / "sentence_chrf_oracle_winner_counts.csv",
    index=False,
    encoding="utf-8-sig",
)

print(
    "Reference-leaking per-turn ceilings:"
)

display(sentence_oracle_summary_df)

print(
    "\nStable sentence-oracle expert usage:"
)

display(
    sentence_chrf_oracle_df[
        "selected_variant"
    ]
    .value_counts()
    .rename_axis("variant")
    .reset_index(name="wins")
)

Reference-leaking per-turn ceilings:


,system,macro_spBLEU,macro_chrFpp,countries,turns,delta_vs_current_spBLEU,delta_vs_current_chrFpp
0,oracle_sentence_chrfpp,35.527641,49.792009,11,12250,4.599638,4.197483
1,oracle_sentence_spbleu,35.897330,49.403448,11,12250,4.969327,3.808922



Stable sentence-oracle expert usage:


,variant,wins
0,00_previous_official_control,3620
1,07_ckpt16000_retrieved_two_shot,2377
2,01_exact_training_parity,1700
3,06_ckpt16500_retrieved_two_shot,1512
4,03_retrieved_two_shot,985
5,02_metadata_no_shots,878
6,04_training_parity_with_participants,593
7,05_retrieved_two_shot_with_participants,585


### **Metadata-routing oracle ladder**

In [8]:
# ============================================================
# Cell 8 — Fast exact group-routing oracle
#
# Replaces the previous Cell 8 completely.
#
# Improvements:
#   1. Extracts flores200 BLEU statistics once.
#   2. Reuses cached sufficient statistics.
#   3. Optimizes complete-country corpus spBLEU.
#   4. Does not independently optimize isolated group BLEU.
#   5. Uses hierarchical fallback:
#        country-domain-turn -> country-domain -> country
#   6. Keeps best_variant_for_subset() for Cell 9.
#   7. Creates the same BLEU cache reused by Cell 10.
#
# DEV references are still used, so outputs remain diagnostic.
# ============================================================

ROUTE_MAX_PASSES = 6
ROUTE_TOLERANCE = 1e-12
ROUTE_MIN_FINE_GROUP_ROWS = 25

print("=" * 88)
print("FAST EXACT GROUP-ROUTING ORACLE")
print("=" * 88)
print("First execution extracts BLEU statistics once.")
print("Later executions reuse the cache.")
print("No checkpoint, GPU, inference, or internet access is used.")

# ------------------------------------------------------------
# 1. Validate state from Cells 2–7
# ------------------------------------------------------------

required_objects = [
    "analysis_df",
    "candidate_matrix",
    "CANDIDATE_VARIANTS",
    "PREDICTION_COLUMNS",
    "VARIANT_TO_INDEX",
    "CHECKPOINT_BY_VARIANT",
    "COUNTRY_BEST_VARIANT",
    "candidate_country_scores_df",
    "current_summary",
    "sentence_cache_fingerprint",
    "ORACLE_DIR",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "The kernel is missing state from previous cells. "
        f"Missing: {missing_objects}. "
        "Rerun Cells 2–7 first."
    )

if len(analysis_df) != EXPECTED_DEV_TURNS:
    raise RuntimeError(
        f"Expected {EXPECTED_DEV_TURNS} rows, "
        f"found {len(analysis_df)}."
    )

if candidate_matrix.shape != (
    len(analysis_df),
    len(CANDIDATE_VARIANTS),
):
    raise RuntimeError(
        "candidate_matrix shape mismatch: "
        f"{candidate_matrix.shape}"
    )

if not analysis_df.index.equals(
    pd.RangeIndex(len(analysis_df))
):
    analysis_df = (
        analysis_df
        .reset_index(drop=True)
    )

    candidate_matrix = (
        analysis_df[
            PREDICTION_COLUMNS
        ]
        .astype(str)
        .to_numpy()
    )

print("Kernel state validated.")
print("DEV rows:", len(analysis_df))
print("Candidates:", len(CANDIDATE_VARIANTS))

# ------------------------------------------------------------
# 2. Initialize reusable metrics
# ------------------------------------------------------------

corpus_bleu_metric = BLEU(
    tokenize="flores200",
    smooth_method="exp",
    effective_order=False,
)

corpus_chrf_metric = CHRF(
    word_order=2
)

# ------------------------------------------------------------
# 3. Create/load exact per-sentence BLEU statistics
#
# Shape:
#   candidates × turns × 10 BLEU statistics
#
# This uses the same paths and fingerprint as Cell 10.
# ------------------------------------------------------------

statistics_cache_path = (
    ORACLE_DIR
    / "candidate_flores200_bleu_stats.npz"
)

statistics_manifest_path = (
    ORACLE_DIR
    / "candidate_flores200_bleu_stats_manifest.json"
)

statistics_payload = {
    "sentence_cache_fingerprint": (
        sentence_cache_fingerprint
    ),
    "candidates": CANDIDATE_VARIANTS,
    "sacrebleu_version": (
        sacrebleu.__version__
    ),
    "tokenizer": "flores200",
    "effective_order": False,
}

statistics_fingerprint = hashlib.sha256(
    json.dumps(
        statistics_payload,
        sort_keys=True,
    ).encode("utf-8")
).hexdigest()

expected_statistics_shape = (
    len(CANDIDATE_VARIANTS),
    len(analysis_df),
    10,
)

use_statistics_cache = False
candidate_bleu_statistics = None

if (
    statistics_cache_path.exists()
    and statistics_manifest_path.exists()
):
    try:
        with open(
            statistics_manifest_path,
            "r",
            encoding="utf-8",
        ) as handle:
            existing_statistics_manifest = (
                json.load(handle)
            )

        manifest_matches = (
            existing_statistics_manifest.get(
                "fingerprint"
            )
            == statistics_fingerprint
        )

        if manifest_matches:
            loaded_statistics = np.load(
                statistics_cache_path
            )["stats"]

            if (
                loaded_statistics.shape
                == expected_statistics_shape
            ):
                candidate_bleu_statistics = (
                    loaded_statistics
                )

                use_statistics_cache = True

    except Exception as cache_error:
        print(
            "Existing statistics cache could not be used:"
        )
        print(
            type(cache_error).__name__,
            str(cache_error),
        )

if use_statistics_cache:
    print(
        "\nLoaded cached BLEU statistics:"
    )
    print(statistics_cache_path)

else:
    print(
        "\nExtracting candidate BLEU statistics..."
    )
    print(
        "This is the only SentencePiece-heavy stage."
    )

    references = (
        analysis_df[
            "reference_arabic"
        ]
        .astype(str)
        .tolist()
    )

    all_candidate_statistics = []

    for variant_name in tqdm(
        CANDIDATE_VARIANTS,
        total=len(CANDIDATE_VARIANTS),
        desc="Tokenize candidates once",
    ):
        predictions = (
            analysis_df[
                f"prediction__{variant_name}"
            ]
            .astype(str)
            .tolist()
        )

        variant_statistics = (
            corpus_bleu_metric
            ._extract_corpus_statistics(
                predictions,
                [references],
            )
        )

        variant_statistics = np.asarray(
            variant_statistics,
            dtype=np.int64,
        )

        if variant_statistics.shape != (
            len(analysis_df),
            10,
        ):
            raise RuntimeError(
                f"Unexpected statistics shape for "
                f"{variant_name}: "
                f"{variant_statistics.shape}"
            )

        all_candidate_statistics.append(
            variant_statistics
        )

    candidate_bleu_statistics = np.stack(
        all_candidate_statistics,
        axis=0,
    )

    np.savez_compressed(
        statistics_cache_path,
        stats=candidate_bleu_statistics,
    )

    with open(
        statistics_manifest_path,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            {
                **statistics_payload,
                "fingerprint": (
                    statistics_fingerprint
                ),
                "shape": list(
                    candidate_bleu_statistics
                    .shape
                ),
                "created_at": time.strftime(
                    "%Y-%m-%d %H:%M:%S"
                ),
            },
            handle,
            ensure_ascii=False,
            indent=2,
        )

    print(
        "Saved BLEU statistics cache:"
    )
    print(statistics_cache_path)

if (
    candidate_bleu_statistics.shape
    != expected_statistics_shape
):
    raise RuntimeError(
        "Final BLEU statistics shape mismatch: "
        f"{candidate_bleu_statistics.shape}"
    )

# ------------------------------------------------------------
# 4. Fast BLEU helpers
# ------------------------------------------------------------

def score_bleu_statistics(
    aggregated_statistics,
):
    return float(
        corpus_bleu_metric
        ._compute_score_from_stats(
            np.asarray(
                aggregated_statistics
            ).tolist()
        )
        .score
    )


def candidate_score_on_indices(
    candidate_index,
    row_indices,
):
    row_indices = np.asarray(
        row_indices,
        dtype=np.int64,
    )

    aggregated_statistics = (
        candidate_bleu_statistics[
            candidate_index,
            row_indices,
            :,
        ]
        .sum(axis=0)
    )

    return score_bleu_statistics(
        aggregated_statistics
    )


def selected_score_on_indices(
    selected_candidate_indices,
    row_indices,
):
    row_indices = np.asarray(
        row_indices,
        dtype=np.int64,
    )

    selected_candidate_indices = (
        np.asarray(
            selected_candidate_indices,
            dtype=np.int16,
        )
    )

    aggregated_statistics = (
        candidate_bleu_statistics[
            selected_candidate_indices,
            row_indices,
            :,
        ]
        .sum(axis=0)
    )

    return score_bleu_statistics(
        aggregated_statistics
    )


# ------------------------------------------------------------
# 5. Validate cached statistics against Cell 4
# ------------------------------------------------------------

candidate_country_score_lookup = (
    candidate_country_scores_df
    .set_index(
        ["system", "country"]
    )["spBLEU"]
)

country_index_map = {
    country: np.flatnonzero(
        analysis_df[
            "config"
        ]
        .astype(str)
        .eq(country)
        .to_numpy()
    )
    for country in sorted(
        analysis_df[
            "config"
        ]
        .astype(str)
        .unique()
    )
}

maximum_statistics_drift = 0.0

for candidate_index, variant_name in enumerate(
    CANDIDATE_VARIANTS
):
    for country, row_indices in (
        country_index_map.items()
    ):
        statistics_score = (
            candidate_score_on_indices(
                candidate_index,
                row_indices,
            )
        )

        direct_score = float(
            candidate_country_score_lookup.loc[
                (
                    variant_name,
                    country,
                )
            ]
        )

        maximum_statistics_drift = max(
            maximum_statistics_drift,
            abs(
                statistics_score
                - direct_score
            ),
        )

if maximum_statistics_drift > 1e-8:
    raise RuntimeError(
        "Cached statistics do not reproduce "
        "the official candidate scores. "
        f"Maximum drift: "
        f"{maximum_statistics_drift}"
    )

print(
    "\nStatistics validation passed."
)
print(
    "Maximum official-score drift:",
    f"{maximum_statistics_drift:.12g}",
)

# ------------------------------------------------------------
# 6. Fast subset selector retained for Cell 9
#
# Cell 9 calls this function when fitting routes on its
# training folds. It now uses cached statistics rather than
# repeatedly tokenizing every candidate.
# ------------------------------------------------------------

def best_variant_for_subset(
    subset_df,
):
    row_indices = (
        subset_df.index
        .to_numpy(dtype=np.int64)
    )

    if len(row_indices) == 0:
        raise ValueError(
            "Cannot select a candidate "
            "for an empty subset."
        )

    candidate_scores = np.asarray([
        candidate_score_on_indices(
            candidate_index,
            row_indices,
        )
        for candidate_index in range(
            len(CANDIDATE_VARIANTS)
        )
    ])

    best_spbleu = float(
        candidate_scores.max()
    )

    tied_candidate_indices = (
        np.flatnonzero(
            np.isclose(
                candidate_scores,
                best_spbleu,
                rtol=0.0,
                atol=1e-12,
            )
        )
    )

    # chrF++ is calculated only for an exact BLEU tie.
    best_chrfpp = np.nan

    if len(tied_candidate_indices) == 1:
        best_candidate_index = int(
            tied_candidate_indices[0]
        )

    else:
        references = (
            subset_df[
                "reference_arabic"
            ]
            .astype(str)
            .tolist()
        )

        tie_rows = []

        for candidate_index in (
            tied_candidate_indices
        ):
            variant_name = (
                CANDIDATE_VARIANTS[
                    candidate_index
                ]
            )

            predictions = (
                subset_df[
                    f"prediction__{variant_name}"
                ]
                .astype(str)
                .tolist()
            )

            chrfpp = float(
                corpus_chrf_metric
                .corpus_score(
                    predictions,
                    [references],
                )
                .score
            )

            tie_rows.append((
                chrfpp,
                variant_name,
                int(candidate_index),
            ))

        tie_rows = sorted(
            tie_rows,
            key=lambda row: (
                -row[0],
                row[1],
            ),
        )

        best_chrfpp = float(
            tie_rows[0][0]
        )

        best_candidate_index = int(
            tie_rows[0][2]
        )

    return {
        "variant": (
            CANDIDATE_VARIANTS[
                best_candidate_index
            ]
        ),
        "spBLEU": float(
            candidate_scores[
                best_candidate_index
            ]
        ),
        "chrF++": best_chrfpp,
    }


# ------------------------------------------------------------
# 7. Build group information for one country
# ------------------------------------------------------------

def prepare_country_groups(
    country_indices,
    group_columns,
    min_rows,
):
    country_indices = np.asarray(
        country_indices,
        dtype=np.int64,
    )

    country_df = analysis_df.loc[
        country_indices
    ]

    groupby_key = (
        group_columns[0]
        if len(group_columns) == 1
        else group_columns
    )

    raw_groups = country_df.groupby(
        groupby_key,
        sort=True,
        dropna=False,
    ).groups

    prepared_groups = []

    for raw_key, global_row_indices in (
        raw_groups.items()
    ):
        global_row_indices = np.asarray(
            list(global_row_indices),
            dtype=np.int64,
        )

        local_positions = np.searchsorted(
            country_indices,
            global_row_indices,
        )

        key_values = (
            raw_key
            if isinstance(raw_key, tuple)
            else (raw_key,)
        )

        group_statistics = (
            candidate_bleu_statistics[
                :,
                global_row_indices,
                :,
            ]
            .sum(axis=1)
        )

        prepared_groups.append({
            "key_values": key_values,
            "global_indices": (
                global_row_indices
            ),
            "local_positions": (
                local_positions
            ),
            "rows": len(
                global_row_indices
            ),
            "eligible": (
                len(global_row_indices)
                >= min_rows
            ),
            "candidate_statistics": (
                group_statistics
            ),
        })

    return prepared_groups


# ------------------------------------------------------------
# 8. Coordinate-optimize complete group assignments
#
# Every trial switch is evaluated against the complete
# country's aggregated corpus statistics.
# ------------------------------------------------------------

def optimize_country_group_assignments(
    country,
    country_indices,
    prepared_groups,
    parent_selected_global,
    route_seed,
):
    country_indices = np.asarray(
        country_indices,
        dtype=np.int64,
    )

    parent_selected_local = (
        np.asarray(
            parent_selected_global[
                country_indices
            ],
            dtype=np.int16,
        )
        .copy()
    )

    eligible_groups = [
        group
        for group in prepared_groups
        if group["eligible"]
    ]

    # No eligible fine group: inherit parent exactly.
    if not eligible_groups:
        inherited_score = (
            selected_score_on_indices(
                parent_selected_local,
                country_indices,
            )
        )

        return {
            "selected": (
                parent_selected_local
            ),
            "best_start": (
                "parent_only"
            ),
            "initial_spBLEU": (
                inherited_score
            ),
            "final_spBLEU": (
                inherited_score
            ),
            "moves": 0,
            "passes": 0,
            "trace": [{
                "country": country,
                "start": "parent_only",
                "initial_spBLEU": (
                    inherited_score
                ),
                "final_spBLEU": (
                    inherited_score
                ),
                "moves": 0,
                "passes": 0,
            }],
        }

    # --------------------------------------------------------
    # Construct multiple starts
    # --------------------------------------------------------

    inherited_assignments = []

    for group in eligible_groups:
        parent_values = (
            parent_selected_local[
                group[
                    "local_positions"
                ]
            ]
        )

        parent_variant = int(
            np.bincount(
                parent_values,
                minlength=len(
                    CANDIDATE_VARIANTS
                ),
            ).argmax()
        )

        inherited_assignments.append(
            parent_variant
        )

    local_best_assignments = []

    for group in eligible_groups:
        local_candidate_scores = [
            score_bleu_statistics(
                group[
                    "candidate_statistics"
                ][candidate_index]
            )
            for candidate_index in range(
                len(CANDIDATE_VARIANTS)
            )
        ]

        local_best_assignments.append(
            int(
                np.argmax(
                    local_candidate_scores
                )
            )
        )

    start_assignments = {
        "inherited_parent": (
            inherited_assignments
        ),
        "local_group_best": (
            local_best_assignments
        ),
    }

    for (
        candidate_index,
        variant_name,
    ) in enumerate(CANDIDATE_VARIANTS):
        start_assignments[
            f"constant__{variant_name}"
        ] = [
            candidate_index
            for _ in eligible_groups
        ]

    best_result = None
    trace_rows = []

    for start_number, (
        start_name,
        group_assignments,
    ) in enumerate(
        start_assignments.items()
    ):
        selected_local = (
            parent_selected_local.copy()
        )

        # Eligible groups become one-expert groups.
        # Ineligible groups remain inherited from the parent.
        for group, candidate_index in zip(
            eligible_groups,
            group_assignments,
        ):
            selected_local[
                group[
                    "local_positions"
                ]
            ] = candidate_index

        total_statistics = (
            candidate_bleu_statistics[
                selected_local,
                country_indices,
                :,
            ]
            .sum(axis=0)
        )

        initial_score = (
            score_bleu_statistics(
                total_statistics
            )
        )

        current_score = initial_score

        rng = np.random.default_rng(
            route_seed
            + 97 * start_number
        )

        total_moves = 0
        passes_run = 0

        group_positions = np.arange(
            len(eligible_groups),
            dtype=np.int64,
        )

        for pass_index in range(
            ROUTE_MAX_PASSES
        ):
            pass_moves = 0

            for group_position in (
                rng.permutation(
                    group_positions
                )
            ):
                group = eligible_groups[
                    group_position
                ]

                local_positions = group[
                    "local_positions"
                ]

                current_values = np.unique(
                    selected_local[
                        local_positions
                    ]
                )

                if len(current_values) != 1:
                    raise RuntimeError(
                        "An eligible routing group "
                        "contains multiple experts."
                    )

                current_candidate_index = int(
                    current_values[0]
                )

                base_statistics = (
                    total_statistics
                    - group[
                        "candidate_statistics"
                    ][
                        current_candidate_index
                    ]
                )

                best_candidate_index = (
                    current_candidate_index
                )

                best_candidate_score = (
                    current_score
                )

                for candidate_index in range(
                    len(CANDIDATE_VARIANTS)
                ):
                    if (
                        candidate_index
                        == current_candidate_index
                    ):
                        continue

                    trial_statistics = (
                        base_statistics
                        + group[
                            "candidate_statistics"
                        ][candidate_index]
                    )

                    trial_score = (
                        score_bleu_statistics(
                            trial_statistics
                        )
                    )

                    if (
                        trial_score
                        > best_candidate_score
                        + ROUTE_TOLERANCE
                    ):
                        best_candidate_score = (
                            trial_score
                        )

                        best_candidate_index = (
                            candidate_index
                        )

                if (
                    best_candidate_index
                    != current_candidate_index
                ):
                    selected_local[
                        local_positions
                    ] = best_candidate_index

                    total_statistics = (
                        base_statistics
                        + group[
                            "candidate_statistics"
                        ][
                            best_candidate_index
                        ]
                    )

                    current_score = (
                        best_candidate_score
                    )

                    pass_moves += 1
                    total_moves += 1

            passes_run = pass_index + 1

            if pass_moves == 0:
                break

        final_score = (
            score_bleu_statistics(
                total_statistics
            )
        )

        trace_row = {
            "country": country,
            "start": start_name,
            "initial_spBLEU": (
                initial_score
            ),
            "final_spBLEU": (
                final_score
            ),
            "moves": total_moves,
            "passes": passes_run,
        }

        trace_rows.append(trace_row)

        if (
            best_result is None
            or final_score
            > best_result[
                "final_spBLEU"
            ]
            + ROUTE_TOLERANCE
        ):
            best_result = {
                "selected": (
                    selected_local.copy()
                ),
                "best_start": (
                    start_name
                ),
                "initial_spBLEU": (
                    initial_score
                ),
                "final_spBLEU": (
                    final_score
                ),
                "moves": total_moves,
                "passes": passes_run,
            }

    best_result["trace"] = trace_rows

    return best_result


# ------------------------------------------------------------
# 9. Score a routed output without re-tokenizing BLEU
# ------------------------------------------------------------

def fast_score_routed_selection(
    selected_candidate_indices,
    system_name,
):
    selected_candidate_indices = (
        np.asarray(
            selected_candidate_indices,
            dtype=np.int16,
        )
    )

    country_rows = []

    for country, row_indices in (
        country_index_map.items()
    ):
        country_selected = (
            selected_candidate_indices[
                row_indices
            ]
        )

        aggregated_statistics = (
            candidate_bleu_statistics[
                country_selected,
                row_indices,
                :,
            ]
            .sum(axis=0)
        )

        spbleu = score_bleu_statistics(
            aggregated_statistics
        )

        predictions = (
            candidate_matrix[
                row_indices,
                country_selected,
            ]
            .astype(str)
            .tolist()
        )

        references = (
            analysis_df.loc[
                row_indices,
                "reference_arabic",
            ]
            .astype(str)
            .tolist()
        )

        chrfpp = float(
            corpus_chrf_metric
            .corpus_score(
                predictions,
                [references],
            )
            .score
        )

        country_rows.append({
            "system": system_name,
            "country": country,
            "turns": len(
                row_indices
            ),
            "spBLEU": float(
                spbleu
            ),
            "chrF++": float(
                chrfpp
            ),
        })

    country_scores = pd.DataFrame(
        country_rows
    )

    summary = {
        "system": system_name,
        "macro_spBLEU": float(
            country_scores[
                "spBLEU"
            ].mean()
        ),
        "macro_chrFpp": float(
            country_scores[
                "chrF++"
            ].mean()
        ),
        "countries": len(
            country_scores
        ),
        "turns": int(
            country_scores[
                "turns"
            ].sum()
        ),
    }

    return summary, country_scores


# ------------------------------------------------------------
# 10. Build one exact routing level
# ------------------------------------------------------------

def build_exact_group_route(
    route_name,
    group_columns,
    min_rows,
    parent_selected_variants,
    route_number,
):
    parent_selected_variants = (
        pd.Series(
            parent_selected_variants,
            index=analysis_df.index,
            dtype="object",
        )
    )

    if parent_selected_variants.isna().any():
        raise RuntimeError(
            f"Missing parent assignments "
            f"for {route_name}."
        )

    parent_selected_indices = (
        parent_selected_variants
        .map(VARIANT_TO_INDEX)
        .astype(int)
        .to_numpy()
    )

    if np.isnan(
        parent_selected_indices
        .astype(float)
    ).any():
        raise RuntimeError(
            f"Unknown parent candidate "
            f"in {route_name}."
        )

    final_selected_indices = (
        parent_selected_indices
        .astype(np.int16)
        .copy()
    )

    route_trace_rows = []
    route_group_rows = []

    progress = tqdm(
        country_index_map.items(),
        total=len(country_index_map),
        desc=route_name,
    )

    for country_number, (
        country,
        country_indices,
    ) in enumerate(progress):
        prepared_groups = (
            prepare_country_groups(
                country_indices=(
                    country_indices
                ),
                group_columns=(
                    group_columns
                ),
                min_rows=min_rows,
            )
        )

        result = (
            optimize_country_group_assignments(
                country=country,
                country_indices=(
                    country_indices
                ),
                prepared_groups=(
                    prepared_groups
                ),
                parent_selected_global=(
                    parent_selected_indices
                ),
                route_seed=(
                    SEED
                    + route_number * 100_003
                    + country_number * 1_009
                ),
            )
        )

        final_selected_indices[
            country_indices
        ] = result["selected"]

        for trace_row in result[
            "trace"
        ]:
            route_trace_rows.append({
                "route": route_name,
                **trace_row,
            })

        for group in prepared_groups:
            group_indices = group[
                "global_indices"
            ]

            final_variants = np.unique(
                final_selected_indices[
                    group_indices
                ]
            )

            parent_variants = np.unique(
                parent_selected_indices[
                    group_indices
                ]
            )

            selected_variant_label = (
                CANDIDATE_VARIANTS[
                    int(final_variants[0])
                ]
                if len(final_variants) == 1
                else "mixed_parent"
            )

            parent_variant_label = (
                CANDIDATE_VARIANTS[
                    int(parent_variants[0])
                ]
                if len(parent_variants) == 1
                else "mixed_parent"
            )

            group_row = {
                "route": route_name,
                "country": country,
                "rows": group["rows"],
                "eligible": (
                    group["eligible"]
                ),
                "selection_source": (
                    "coordinate_full_country_spBLEU"
                    if group["eligible"]
                    else "hierarchical_parent_fallback"
                ),
                "parent_variant": (
                    parent_variant_label
                ),
                "selected_variant": (
                    selected_variant_label
                ),
            }

            group_row.update(
                dict(
                    zip(
                        group_columns,
                        group[
                            "key_values"
                        ],
                    )
                )
            )

            route_group_rows.append(
                group_row
            )

        progress.set_postfix({
            "country": country,
            "spBLEU": (
                f"{result['final_spBLEU']:.3f}"
            ),
        })

    route_df = analysis_df[
        BASE_COLUMNS
    ].copy()

    route_df[
        "selected_variant"
    ] = [
        CANDIDATE_VARIANTS[index]
        for index
        in final_selected_indices
    ]

    route_df[
        "selected_checkpoint"
    ] = (
        route_df[
            "selected_variant"
        ].map(
            CHECKPOINT_BY_VARIANT
        )
    )

    route_df[
        "prediction"
    ] = (
        candidate_matrix[
            np.arange(
                len(analysis_df)
            ),
            final_selected_indices,
        ]
    )

    (
        route_summary,
        route_country_scores,
    ) = fast_score_routed_selection(
        selected_candidate_indices=(
            final_selected_indices
        ),
        system_name=route_name,
    )

    route_summary[
        "delta_vs_current_spBLEU"
    ] = (
        route_summary[
            "macro_spBLEU"
        ]
        - current_summary[
            "macro_spBLEU"
        ]
    )

    route_group_df = pd.DataFrame(
        route_group_rows
    )

    country_final_score_map = dict(
        zip(
            route_country_scores[
                "country"
            ],
            route_country_scores[
                "spBLEU"
            ],
        )
    )

    route_group_df[
        "country_final_spBLEU"
    ] = (
        route_group_df[
            "country"
        ].map(
            country_final_score_map
        )
    )

    route_df.to_csv(
        ORACLE_DIR
        / (
            f"{route_name}_"
            "diagnostic_predictions.csv"
        ),
        index=False,
        encoding="utf-8-sig",
    )

    route_group_df.to_csv(
        ORACLE_DIR
        / f"{route_name}_route_map.csv",
        index=False,
        encoding="utf-8-sig",
    )

    pd.DataFrame(
        route_trace_rows
    ).to_csv(
        ORACLE_DIR
        / f"{route_name}_coordinate_trace.csv",
        index=False,
        encoding="utf-8-sig",
    )

    route_country_scores.to_csv(
        ORACLE_DIR
        / f"{route_name}_per_country.csv",
        index=False,
        encoding="utf-8-sig",
    )

    print(
        f"\n{route_name}: "
        f"{route_summary['macro_spBLEU']:.6f} "
        f"spBLEU / "
        f"{route_summary['macro_chrFpp']:.6f} "
        f"chrF++"
    )

    return (
        route_df,
        route_summary,
        route_country_scores,
        route_group_df,
    )


# ------------------------------------------------------------
# 11. Run hierarchy
# ------------------------------------------------------------

reference_route_outputs = {}
reference_route_summaries = []
reference_route_country_parts = []
reference_route_group_maps = {}

# Initial country parent from Cell 4.
initial_country_parent = (
    analysis_df["config"]
    .map(COUNTRY_BEST_VARIANT)
)

# Level 1: one expert per country.
(
    country_route_df,
    country_route_summary,
    country_route_country_scores,
    country_route_group_map,
) = build_exact_group_route(
    route_name=(
        "oracle_route_country"
    ),
    group_columns=[
        "config",
    ],
    min_rows=1,
    parent_selected_variants=(
        initial_country_parent
    ),
    route_number=1,
)

reference_route_outputs[
    "oracle_route_country"
] = country_route_df

reference_route_summaries.append(
    country_route_summary
)

reference_route_country_parts.append(
    country_route_country_scores
)

reference_route_group_maps[
    "oracle_route_country"
] = country_route_group_map

# Level 2A: country × turn bucket.
(
    country_turn_route_df,
    country_turn_route_summary,
    country_turn_country_scores,
    country_turn_group_map,
) = build_exact_group_route(
    route_name=(
        "oracle_route_country_turn"
    ),
    group_columns=[
        "config",
        "turn_bucket",
    ],
    min_rows=(
        ROUTE_MIN_FINE_GROUP_ROWS
    ),
    parent_selected_variants=(
        country_route_df[
            "selected_variant"
        ]
    ),
    route_number=2,
)

reference_route_outputs[
    "oracle_route_country_turn"
] = country_turn_route_df

reference_route_summaries.append(
    country_turn_route_summary
)

reference_route_country_parts.append(
    country_turn_country_scores
)

reference_route_group_maps[
    "oracle_route_country_turn"
] = country_turn_group_map

# Level 2B and Level 3 require domain metadata.
if "domain" in analysis_df.columns:
    (
        country_domain_route_df,
        country_domain_route_summary,
        country_domain_country_scores,
        country_domain_group_map,
    ) = build_exact_group_route(
        route_name=(
            "oracle_route_country_domain"
        ),
        group_columns=[
            "config",
            "domain",
        ],
        min_rows=(
            ROUTE_MIN_FINE_GROUP_ROWS
        ),
        parent_selected_variants=(
            country_route_df[
                "selected_variant"
            ]
        ),
        route_number=3,
    )

    reference_route_outputs[
        "oracle_route_country_domain"
    ] = country_domain_route_df

    reference_route_summaries.append(
        country_domain_route_summary
    )

    reference_route_country_parts.append(
        country_domain_country_scores
    )

    reference_route_group_maps[
        "oracle_route_country_domain"
    ] = country_domain_group_map

    # Hierarchical fallback:
    # small country-domain-turn groups inherit
    # their country-domain route.
    (
        country_domain_turn_route_df,
        country_domain_turn_route_summary,
        country_domain_turn_country_scores,
        country_domain_turn_group_map,
    ) = build_exact_group_route(
        route_name=(
            "oracle_route_country_domain_turn"
        ),
        group_columns=[
            "config",
            "domain",
            "turn_bucket",
        ],
        min_rows=(
            ROUTE_MIN_FINE_GROUP_ROWS
        ),
        parent_selected_variants=(
            country_domain_route_df[
                "selected_variant"
            ]
        ),
        route_number=4,
    )

    reference_route_outputs[
        "oracle_route_country_domain_turn"
    ] = country_domain_turn_route_df

    reference_route_summaries.append(
        country_domain_turn_route_summary
    )

    reference_route_country_parts.append(
        country_domain_turn_country_scores
    )

    reference_route_group_maps[
        "oracle_route_country_domain_turn"
    ] = country_domain_turn_group_map

else:
    print(
        "\nDomain metadata is unavailable. "
        "Country-domain routes were skipped."
    )

# ------------------------------------------------------------
# 12. Final Cell 8 summary
# ------------------------------------------------------------

reference_route_summary_df = (
    pd.DataFrame(
        reference_route_summaries
    )
    .sort_values(
        "macro_spBLEU",
        ascending=False,
    )
    .reset_index(drop=True)
)

reference_route_summary_df.to_csv(
    ORACLE_DIR
    / "reference_selected_route_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

all_reference_route_country_df = (
    pd.concat(
        reference_route_country_parts,
        ignore_index=True,
    )
)

all_reference_route_country_df.to_csv(
    ORACLE_DIR
    / "reference_selected_routes_per_country.csv",
    index=False,
    encoding="utf-8-sig",
)

print("\n" + "=" * 88)
print("CELL 8 COMPLETED")
print("=" * 88)

print(
    "The following scores use DEV references "
    "and remain diagnostic:"
)

display(
    reference_route_summary_df[
        [
            "system",
            "macro_spBLEU",
            "macro_chrFpp",
            "delta_vs_current_spBLEU",
        ]
    ]
)

print(
    "\nBLEU statistics are cached for Cell 10:"
)
print(statistics_cache_path)

print(
    "\nYou can now run Cell 9."
)

FAST EXACT GROUP-ROUTING ORACLE
First execution extracts BLEU statistics once.
Later executions reuse the cache.
No checkpoint, GPU, inference, or internet access is used.
Kernel state validated.
DEV rows: 12250
Candidates: 8

Extracting candidate BLEU statistics...
This is the only SentencePiece-heavy stage.


Tokenize candidates once:   0%|          | 0/8 [00:00<?, ?it/s]

Saved BLEU statistics cache:
/home/mabdallah/alexandriax_mt_14d/inference_variants/_oracle_analysis_v1/candidate_flores200_bleu_stats.npz

Statistics validation passed.
Maximum official-score drift: 0


oracle_route_country:   0%|          | 0/11 [00:00<?, ?it/s]


oracle_route_country: 30.928003 spBLEU / 45.594526 chrF++


...


oracle_route_country_domain: 31.399517 spBLEU / 45.896135 chrF++


oracle_route_country_domain_turn:   0%|          | 0/11 [00:00<?, ?it/s]


oracle_route_country_domain_turn: 31.842600 spBLEU / 46.202679 chrF++

CELL 8 COMPLETED
The following scores use DEV references and remain diagnostic:


,system,macro_spBLEU,macro_chrFpp,delta_vs_current_spBLEU
0,oracle_route_country_domain_turn,31.842600,46.202679,0.914597
1,oracle_route_country_domain,31.399517,45.896135,0.471514
2,oracle_route_country_turn,31.125025,45.704114,0.197022
3,oracle_route_country,30.928003,45.594526,0.000000



BLEU statistics are cached for Cell 10:
/home/mabdallah/alexandriax_mt_14d/inference_variants/_oracle_analysis_v1/candidate_flores200_bleu_stats.npz

You can now run Cell 9.


### **Honest out-of-fold static routing**

In [9]:
# ============================================================
# Cell 9 — Conversation-level out-of-fold routing
#
# The selected expert for a held-out conversation is learned
# only from the other folds.
# ============================================================

def best_macro_variant(subset_df):
    candidate_rows = []

    for variant_name in CANDIDATE_VARIANTS:
        country_spbleu = []
        country_chrfpp = []

        for _, country_df in (
            subset_df.groupby(
                "config",
                sort=True,
            )
        ):
            predictions = (
                country_df[
                    f"prediction__{variant_name}"
                ]
                .astype(str)
                .tolist()
            )

            references = (
                country_df[
                    "reference_arabic"
                ]
                .astype(str)
                .tolist()
            )

            country_spbleu.append(
                float(
                    sacrebleu.corpus_bleu(
                        predictions,
                        [references],
                        tokenize="flores200",
                    ).score
                )
            )

            country_chrfpp.append(
                float(
                    sacrebleu.corpus_chrf(
                        predictions,
                        [references],
                        word_order=2,
                    ).score
                )
            )

        candidate_rows.append({
            "variant": variant_name,
            "macro_spBLEU": float(
                np.mean(country_spbleu)
            ),
            "macro_chrFpp": float(
                np.mean(country_chrfpp)
            ),
        })

    return sorted(
        candidate_rows,
        key=lambda row: (
            -row["macro_spBLEU"],
            -row["macro_chrFpp"],
            row["variant"],
        ),
    )[0]


def stable_conversation_fold(
    country,
    conversation_id,
):
    raw = (
        f"{country}|{conversation_id}|{SEED}"
        .encode("utf-8")
    )

    return (
        int(
            hashlib.sha256(raw)
            .hexdigest()[:12],
            16,
        )
        % OOF_FOLDS
    )


analysis_df["oof_fold"] = [
    stable_conversation_fold(
        country,
        conversation_id,
    )
    for country, conversation_id in zip(
        analysis_df["config"],
        analysis_df[
            "conversation_id"
        ],
    )
]

conversation_fold_counts = (
    analysis_df
    .groupby(
        [
            "config",
            "conversation_id",
        ]
    )["oof_fold"]
    .nunique()
)

if conversation_fold_counts.max() != 1:
    raise RuntimeError(
        "A conversation was split "
        "between OOF folds."
    )


def fit_group_lookup(
    train_df,
    group_columns,
    min_rows,
):
    lookup = {}

    groupby_key = (
        group_columns[0]
        if len(group_columns) == 1
        else group_columns
    )

    groups = train_df.groupby(
        groupby_key,
        sort=True,
        dropna=False,
    ).groups

    for raw_key, row_indices in (
        groups.items()
    ):
        subset_df = train_df.loc[
            row_indices
        ]

        if len(subset_df) < min_rows:
            continue

        key = (
            raw_key
            if isinstance(raw_key, tuple)
            else (raw_key,)
        )

        lookup[key] = (
            best_variant_for_subset(
                subset_df
            )["variant"]
        )

    return lookup


oof_route_specs = {
    "oof_global": [],
    "oof_country": [
        ["config"],
    ],
    "oof_country_turn": [
        ["config"],
        ["config", "turn_bucket"],
    ],
}

if "domain" in analysis_df.columns:
    oof_route_specs.update({
        "oof_country_domain": [
            ["config"],
            ["config", "domain"],
        ],
        "oof_country_domain_turn": [
            ["config"],
            ["config", "domain"],
            [
                "config",
                "domain",
                "turn_bucket",
            ],
        ],
    })

oof_selected_variants = {
    route_name: pd.Series(
        index=analysis_df.index,
        dtype="object",
    )
    for route_name in oof_route_specs
}

oof_fit_rows = []

for held_out_fold in range(
    OOF_FOLDS
):
    train_df = analysis_df.loc[
        analysis_df[
            "oof_fold"
        ].ne(held_out_fold)
    ]

    validation_indices = (
        analysis_df.index[
            analysis_df[
                "oof_fold"
            ].eq(held_out_fold)
        ]
    )

    validation_df = analysis_df.loc[
        validation_indices
    ]

    global_variant = (
        best_macro_variant(
            train_df
        )["variant"]
    )

    for route_name, hierarchy in (
        oof_route_specs.items()
    ):
        fitted_levels = []

        for level_index, group_columns in (
            enumerate(hierarchy)
        ):
            min_rows = (
                1
                if level_index == 0
                else OOF_MIN_FINE_GROUP_ROWS
            )

            fitted_levels.append((
                group_columns,
                fit_group_lookup(
                    train_df,
                    group_columns,
                    min_rows,
                ),
            ))

        for row_index, row in (
            validation_df.iterrows()
        ):
            selected_variant = (
                global_variant
            )

            for (
                group_columns,
                lookup,
            ) in fitted_levels:
                key = tuple(
                    row[column]
                    for column
                    in group_columns
                )

                if key in lookup:
                    selected_variant = (
                        lookup[key]
                    )

            oof_selected_variants[
                route_name
            ].at[row_index] = (
                selected_variant
            )

        oof_fit_rows.append({
            "held_out_fold": held_out_fold,
            "route": route_name,
            "train_rows": len(train_df),
            "validation_rows": (
                len(validation_df)
            ),
            "global_fallback": (
                global_variant
            ),
        })

oof_summary_rows = []
oof_country_parts = []

oof_assignment_df = analysis_df[
    [
        "source_id",
        "config",
        "conversation_id",
        "turn_order",
        "oof_fold",
    ]
].copy()

for route_name, selected_series in (
    oof_selected_variants.items()
):
    if selected_series.isna().any():
        raise RuntimeError(
            f"Unrouted OOF rows: "
            f"{route_name}"
        )

    selected_indices = (
        selected_series
        .map(VARIANT_TO_INDEX)
        .astype(int)
        .to_numpy()
    )

    prediction_column = (
        f"prediction__{route_name}"
    )

    analysis_df[prediction_column] = (
        candidate_matrix[
            np.arange(
                len(analysis_df)
            ),
            selected_indices,
        ]
    )

    oof_assignment_df[
        f"selected_variant__{route_name}"
    ] = selected_series

    summary, country_scores = (
        official_macro_scores(
            analysis_df,
            prediction_column,
            route_name,
        )
    )

    summary[
        "delta_vs_current_spBLEU"
    ] = (
        summary["macro_spBLEU"]
        - current_summary[
            "macro_spBLEU"
        ]
    )

    oof_summary_rows.append(
        summary
    )
    oof_country_parts.append(
        country_scores
    )

oof_router_summary_df = (
    pd.DataFrame(
        oof_summary_rows
    )
    .sort_values(
        "macro_spBLEU",
        ascending=False,
    )
    .reset_index(drop=True)
)

oof_router_country_df = pd.concat(
    oof_country_parts,
    ignore_index=True,
)

oof_router_summary_df.to_csv(
    ORACLE_DIR
    / "oof_static_router_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

oof_router_country_df.to_csv(
    ORACLE_DIR
    / "oof_static_router_per_country.csv",
    index=False,
    encoding="utf-8-sig",
)

oof_assignment_df.to_csv(
    ORACLE_DIR
    / "oof_static_router_assignments.csv",
    index=False,
    encoding="utf-8-sig",
)

pd.DataFrame(oof_fit_rows).to_csv(
    ORACLE_DIR
    / "oof_static_router_fit_log.csv",
    index=False,
    encoding="utf-8-sig",
)

print(
    "Honest out-of-fold routing estimates:"
)

display(oof_router_summary_df)

Honest out-of-fold routing estimates:


,system,macro_spBLEU,macro_chrFpp,countries,turns,delta_vs_current_spBLEU
0,oof_country,30.756904,45.492084,11,12250,-0.171099
1,oof_country_turn,30.697849,45.452443,11,12250,-0.230153
2,oof_country_domain_turn,30.671309,45.422187,11,12250,-0.256693
3,oof_country_domain,30.663993,45.401495,11,12250,-0.264009
4,oof_global,30.628726,45.420629,11,12250,-0.299277


### **Exact corpus-spBLEU coordinate oracle**

In [10]:
# ============================================================
# Cell 10 — Exact-corpus-spBLEU coordinate oracle
#
# Each proposed switch is scored using exact aggregated
# flores200 BLEU statistics for that country.
#
# This remains a reference-leaking diagnostic and is not
# a submission system.
# ============================================================

coordinate_oracle_df = None
coordinate_oracle_summary = None
coordinate_oracle_country_scores = None

if not RUN_COORDINATE_ORACLE:
    print(
        "Coordinate oracle skipped. "
        "Set RUN_COORDINATE_ORACLE=True."
    )

else:
    corpus_bleu_metric = BLEU(
        tokenize="flores200",
        smooth_method="exp",
        effective_order=False,
    )

    statistics_cache_path = (
        ORACLE_DIR
        / "candidate_flores200_bleu_stats.npz"
    )

    statistics_manifest_path = (
        ORACLE_DIR
        / "candidate_flores200_bleu_stats_manifest.json"
    )

    statistics_payload = {
        "sentence_cache_fingerprint": (
            sentence_cache_fingerprint
        ),
        "candidates": CANDIDATE_VARIANTS,
        "sacrebleu_version": (
            sacrebleu.__version__
        ),
        "tokenizer": "flores200",
        "effective_order": False,
    }

    statistics_fingerprint = (
        hashlib.sha256(
            json.dumps(
                statistics_payload,
                sort_keys=True,
            ).encode("utf-8")
        ).hexdigest()
    )

    use_statistics_cache = False

    if (
        statistics_cache_path.exists()
        and statistics_manifest_path.exists()
    ):
        with open(
            statistics_manifest_path,
            "r",
            encoding="utf-8",
        ) as handle:
            old_statistics_manifest = (
                json.load(handle)
            )

        use_statistics_cache = (
            old_statistics_manifest.get(
                "fingerprint"
            )
            == statistics_fingerprint
        )

    if use_statistics_cache:
        candidate_bleu_statistics = (
            np.load(
                statistics_cache_path
            )["stats"]
        )

        print(
            "Loaded BLEU statistics:",
            statistics_cache_path,
        )

    else:
        references = (
            analysis_df[
                "reference_arabic"
            ]
            .astype(str)
            .tolist()
        )

        all_statistics = []

        for variant_name in tqdm(
            CANDIDATE_VARIANTS,
            desc="Extract BLEU statistics",
        ):
            predictions = (
                analysis_df[
                    f"prediction__{variant_name}"
                ]
                .astype(str)
                .tolist()
            )

            variant_statistics = (
                corpus_bleu_metric
                ._extract_corpus_statistics(
                    predictions,
                    [references],
                )
            )

            all_statistics.append(
                np.asarray(
                    variant_statistics,
                    dtype=np.int64,
                )
            )

        candidate_bleu_statistics = (
            np.stack(
                all_statistics,
                axis=0,
            )
        )

        np.savez_compressed(
            statistics_cache_path,
            stats=candidate_bleu_statistics,
        )

        with open(
            statistics_manifest_path,
            "w",
            encoding="utf-8",
        ) as handle:
            json.dump(
                {
                    **statistics_payload,
                    "fingerprint": (
                        statistics_fingerprint
                    ),
                    "shape": list(
                        candidate_bleu_statistics
                        .shape
                    ),
                    "created_at": (
                        time.strftime(
                            "%Y-%m-%d %H:%M:%S"
                        )
                    ),
                },
                handle,
                indent=2,
            )

    expected_statistics_shape = (
        len(CANDIDATE_VARIANTS),
        len(analysis_df),
        10,
    )

    if (
        candidate_bleu_statistics.shape
        != expected_statistics_shape
    ):
        raise ValueError(
            "Unexpected BLEU-statistics shape: "
            f"{candidate_bleu_statistics.shape}; "
            f"expected "
            f"{expected_statistics_shape}"
        )

    country_index_map = {
        country: np.flatnonzero(
            analysis_df[
                "config"
            ]
            .astype(str)
            .eq(country)
            .to_numpy()
        )
        for country in sorted(
            analysis_df[
                "config"
            ]
            .astype(str)
            .unique()
        )
    }

    # --------------------------------------------------------
    # Verify that sufficient statistics exactly reproduce
    # the direct official candidate scores.
    # --------------------------------------------------------

    maximum_statistics_drift = 0.0

    candidate_country_lookup = (
        candidate_country_scores_df
        .set_index(
            ["system", "country"]
        )["spBLEU"]
    )

    for variant_index, variant_name in (
        enumerate(CANDIDATE_VARIANTS)
    ):
        for country, row_indices in (
            country_index_map.items()
        ):
            aggregate_statistics = (
                candidate_bleu_statistics[
                    variant_index,
                    row_indices,
                    :,
                ]
                .sum(axis=0)
                .tolist()
            )

            statistics_score = float(
                corpus_bleu_metric
                ._compute_score_from_stats(
                    aggregate_statistics
                )
                .score
            )

            direct_score = float(
                candidate_country_lookup.loc[
                    (
                        variant_name,
                        country,
                    )
                ]
            )

            maximum_statistics_drift = max(
                maximum_statistics_drift,
                abs(
                    statistics_score
                    - direct_score
                ),
            )

    if maximum_statistics_drift > 1e-8:
        raise RuntimeError(
            "BLEU-statistics validation failed. "
            f"Maximum drift: "
            f"{maximum_statistics_drift}"
        )

    print(
        "Exact-statistics validation drift:",
        maximum_statistics_drift,
    )

    def score_aggregated_statistics(
        statistics,
    ):
        return float(
            corpus_bleu_metric
            ._compute_score_from_stats(
                np.asarray(
                    statistics
                ).tolist()
            )
            .score
        )

    def coordinate_optimize_country(
        row_indices,
        initial_selection,
        random_seed,
    ):
        row_indices = np.asarray(
            row_indices,
            dtype=np.int64,
        )

        selected = np.asarray(
            initial_selection,
            dtype=np.int16,
        ).copy()

        local_positions = np.arange(
            len(row_indices),
            dtype=np.int64,
        )

        total_statistics = (
            candidate_bleu_statistics[
                selected,
                row_indices,
                :,
            ]
            .sum(axis=0)
        )

        initial_score = (
            score_aggregated_statistics(
                total_statistics
            )
        )

        current_score = initial_score

        rng = np.random.default_rng(
            random_seed
        )

        total_moves = 0
        passes_run = 0

        for pass_index in range(
            COORDINATE_MAX_PASSES
        ):
            pass_moves = 0

            for local_position in (
                rng.permutation(
                    local_positions
                )
            ):
                global_row = row_indices[
                    local_position
                ]

                old_variant_index = int(
                    selected[
                        local_position
                    ]
                )

                base_statistics = (
                    total_statistics
                    - candidate_bleu_statistics[
                        old_variant_index,
                        global_row,
                        :,
                    ]
                )

                best_variant_index = (
                    old_variant_index
                )

                best_score = current_score

                for candidate_index in range(
                    len(CANDIDATE_VARIANTS)
                ):
                    if (
                        candidate_index
                        == old_variant_index
                    ):
                        continue

                    trial_statistics = (
                        base_statistics
                        + candidate_bleu_statistics[
                            candidate_index,
                            global_row,
                            :,
                        ]
                    )

                    trial_score = (
                        score_aggregated_statistics(
                            trial_statistics
                        )
                    )

                    if (
                        trial_score
                        > best_score
                        + COORDINATE_TOLERANCE
                    ):
                        best_score = trial_score
                        best_variant_index = (
                            candidate_index
                        )

                if (
                    best_variant_index
                    != old_variant_index
                ):
                    total_statistics = (
                        base_statistics
                        + candidate_bleu_statistics[
                            best_variant_index,
                            global_row,
                            :,
                        ]
                    )

                    selected[
                        local_position
                    ] = best_variant_index

                    current_score = best_score
                    pass_moves += 1
                    total_moves += 1

            passes_run = pass_index + 1

            if pass_moves == 0:
                break

        verified_final_score = (
            score_aggregated_statistics(
                total_statistics
            )
        )

        if (
            abs(
                verified_final_score
                - current_score
            )
            > 1e-10
        ):
            raise RuntimeError(
                "Coordinate score bookkeeping drift."
            )

        return {
            "selected": selected,
            "initial_spBLEU": initial_score,
            "final_spBLEU": (
                verified_final_score
            ),
            "moves": total_moves,
            "passes": passes_run,
        }

    # --------------------------------------------------------
    # Multiple starting points
    # --------------------------------------------------------

    current_start = (
        analysis_df[
            "selected_from_variant"
        ]
        .map(VARIANT_TO_INDEX)
        .astype(int)
        .to_numpy()
    )

    sentence_chrf_start = (
        sentence_chrf_oracle_df
        .set_index("source_id")
        .loc[
            analysis_df["source_id"],
            "selected_variant",
        ]
        .map(VARIANT_TO_INDEX)
        .astype(int)
        .to_numpy()
    )

    sentence_bleu_start = (
        sentence_bleu_oracle_df
        .set_index("source_id")
        .loc[
            analysis_df["source_id"],
            "selected_variant",
        ]
        .map(VARIANT_TO_INDEX)
        .astype(int)
        .to_numpy()
    )

    start_vectors = {
        "current_92": current_start,
        "sentence_chrfpp": (
            sentence_chrf_start
        ),
        "sentence_spbleu": (
            sentence_bleu_start
        ),
    }

    for (
        variant_name,
        variant_index,
    ) in VARIANT_TO_INDEX.items():
        start_vectors[
            f"constant__{variant_name}"
        ] = np.full(
            len(analysis_df),
            variant_index,
            dtype=np.int16,
        )

    final_selected_indices = np.empty(
        len(analysis_df),
        dtype=np.int16,
    )

    coordinate_trace_rows = []

    for country_number, (
        country,
        row_indices,
    ) in enumerate(
        tqdm(
            country_index_map.items(),
            desc="Coordinate oracle by country",
        )
    ):
        best_result = None
        best_start_name = None

        for start_number, (
            start_name,
            global_start,
        ) in enumerate(
            start_vectors.items()
        ):
            result = (
                coordinate_optimize_country(
                    row_indices=(
                        row_indices
                    ),
                    initial_selection=(
                        global_start[
                            row_indices
                        ]
                    ),
                    random_seed=(
                        SEED
                        + 1009
                        * country_number
                        + 37
                        * start_number
                    ),
                )
            )

            coordinate_trace_rows.append({
                "country": country,
                "start": start_name,
                "initial_spBLEU": (
                    result[
                        "initial_spBLEU"
                    ]
                ),
                "final_spBLEU": (
                    result[
                        "final_spBLEU"
                    ]
                ),
                "moves": result["moves"],
                "passes": result["passes"],
            })

            if (
                best_result is None
                or result[
                    "final_spBLEU"
                ]
                > best_result[
                    "final_spBLEU"
                ]
                + COORDINATE_TOLERANCE
            ):
                best_result = result
                best_start_name = (
                    start_name
                )

        final_selected_indices[
            row_indices
        ] = best_result["selected"]

        print(
            f"{country}: "
            f"{best_result['final_spBLEU']:.6f} "
            f"from {best_start_name}"
        )

    coordinate_oracle_df = (
        analysis_df[
            BASE_COLUMNS
        ].copy()
    )

    coordinate_oracle_df[
        "selected_variant"
    ] = [
        CANDIDATE_VARIANTS[index]
        for index
        in final_selected_indices
    ]

    coordinate_oracle_df[
        "selected_checkpoint"
    ] = (
        coordinate_oracle_df[
            "selected_variant"
        ].map(CHECKPOINT_BY_VARIANT)
    )

    coordinate_oracle_df[
        "prediction"
    ] = (
        candidate_matrix[
            np.arange(
                len(analysis_df)
            ),
            final_selected_indices,
        ]
    )

    (
        coordinate_oracle_summary,
        coordinate_oracle_country_scores,
    ) = official_macro_scores(
        coordinate_oracle_df,
        "prediction",
        "oracle_coordinate_corpus_spbleu",
    )

    coordinate_oracle_summary[
        "delta_vs_current_spBLEU"
    ] = (
        coordinate_oracle_summary[
            "macro_spBLEU"
        ]
        - current_summary[
            "macro_spBLEU"
        ]
    )

    coordinate_oracle_summary[
        "delta_vs_current_chrFpp"
    ] = (
        coordinate_oracle_summary[
            "macro_chrFpp"
        ]
        - current_summary[
            "macro_chrFpp"
        ]
    )

    coordinate_oracle_df.to_csv(
        ORACLE_DIR
        / (
            "oracle_coordinate_corpus_spbleu_"
            "diagnostic_predictions.csv"
        ),
        index=False,
        encoding="utf-8-sig",
    )

    coordinate_oracle_country_scores.to_csv(
        ORACLE_DIR
        / "coordinate_oracle_per_country.csv",
        index=False,
        encoding="utf-8-sig",
    )

    pd.DataFrame(
        coordinate_trace_rows
    ).to_csv(
        ORACLE_DIR
        / "coordinate_oracle_trace.csv",
        index=False,
        encoding="utf-8-sig",
    )

    print(
        "\nCoordinate corpus-spBLEU oracle:"
    )

    display(
        pd.DataFrame([
            coordinate_oracle_summary
        ])
    )

Loaded BLEU statistics: /home/mabdallah/alexandriax_mt_14d/inference_variants/_oracle_analysis_v1/candidate_flores200_bleu_stats.npz
Exact-statistics validation drift: 0.0


Coordinate oracle by country:   0%|          | 0/11 [00:00<?, ?it/s]

EG: 38.322313 from constant__00_previous_official_control
JO: 41.247103 from constant__06_ckpt16500_retrieved_two_shot
LB: 36.916171 from current_92
MA: 28.489118 from sentence_spbleu
MR: 22.066587 from constant__00_previous_official_control
OM: 41.147652 from current_92
PS: 38.020698 from current_92
SA: 37.952262 from current_92
SY: 45.921520 from sentence_spbleu
TN: 33.772705 from current_92
YE: 32.283325 from current_92

Coordinate corpus-spBLEU oracle:


,system,macro_spBLEU,macro_chrFpp,countries,turns,delta_vs_current_spBLEU,delta_vs_current_chrFpp
0,oracle_coordinate_corpus_spbleu,36.012678,49.378289,11,12250,5.084675,3.783763


### **Final diagnosis and country priorities**

In [11]:
# ============================================================
# Cell 11 — Final diagnosis and next-step decision
# ============================================================

summary_rows = []

best_single_row = (
    candidate_ranking_df.iloc[0]
)

summary_rows.append({
    "analysis_level": (
        "best_single_candidate"
    ),
    "system": best_single_row["system"],
    "macro_spBLEU": (
        best_single_row[
            "macro_spBLEU"
        ]
    ),
    "macro_chrFpp": (
        best_single_row[
            "macro_chrFpp"
        ]
    ),
    "reference_selected": True,
    "honest_oof": False,
})

summary_rows.append({
    "analysis_level": "current_system",
    "system": CURRENT_SYSTEM_VARIANT,
    "macro_spBLEU": (
        current_summary[
            "macro_spBLEU"
        ]
    ),
    "macro_chrFpp": (
        current_summary[
            "macro_chrFpp"
        ]
    ),
    "reference_selected": True,
    "honest_oof": False,
})

for row in (
    reference_route_summary_df
    .to_dict("records")
):
    summary_rows.append({
        "analysis_level": (
            "metadata_group_oracle"
        ),
        "system": row["system"],
        "macro_spBLEU": (
            row["macro_spBLEU"]
        ),
        "macro_chrFpp": (
            row["macro_chrFpp"]
        ),
        "reference_selected": True,
        "honest_oof": False,
    })

for row in (
    oof_router_summary_df
    .to_dict("records")
):
    summary_rows.append({
        "analysis_level": (
            "static_router_oof"
        ),
        "system": row["system"],
        "macro_spBLEU": (
            row["macro_spBLEU"]
        ),
        "macro_chrFpp": (
            row["macro_chrFpp"]
        ),
        "reference_selected": False,
        "honest_oof": True,
    })

for row in (
    sentence_oracle_summary_df
    .to_dict("records")
):
    summary_rows.append({
        "analysis_level": (
            "sentence_oracle"
        ),
        "system": row["system"],
        "macro_spBLEU": (
            row["macro_spBLEU"]
        ),
        "macro_chrFpp": (
            row["macro_chrFpp"]
        ),
        "reference_selected": True,
        "honest_oof": False,
    })

if coordinate_oracle_summary is not None:
    summary_rows.append({
        "analysis_level": (
            "corpus_coordinate_oracle"
        ),
        "system": (
            coordinate_oracle_summary[
                "system"
            ]
        ),
        "macro_spBLEU": (
            coordinate_oracle_summary[
                "macro_spBLEU"
            ]
        ),
        "macro_chrFpp": (
            coordinate_oracle_summary[
                "macro_chrFpp"
            ]
        ),
        "reference_selected": True,
        "honest_oof": False,
    })

oracle_summary_df = pd.DataFrame(
    summary_rows
)

oracle_summary_df[
    "delta_vs_current_spBLEU"
] = (
    oracle_summary_df[
        "macro_spBLEU"
    ]
    - current_summary[
        "macro_spBLEU"
    ]
)

oracle_summary_df[
    "delta_vs_current_chrFpp"
] = (
    oracle_summary_df[
        "macro_chrFpp"
    ]
    - current_summary[
        "macro_chrFpp"
    ]
)

oracle_summary_df = (
    oracle_summary_df
    .sort_values(
        "macro_spBLEU",
        ascending=False,
    )
    .reset_index(drop=True)
)

oracle_summary_df.to_csv(
    ORACLE_DIR
    / "oracle_analysis_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

# ------------------------------------------------------------
# Country-level headroom
# ------------------------------------------------------------

country_headroom_df = (
    current_country_scores[
        [
            "country",
            "turns",
            "spBLEU",
            "chrF++",
        ]
    ]
    .rename(
        columns={
            "spBLEU": (
                "current_spBLEU"
            ),
            "chrF++": (
                "current_chrFpp"
            ),
        }
    )
)

country_headroom_df = (
    country_headroom_df.merge(
        sentence_chrf_country_scores[
            [
                "country",
                "spBLEU",
                "chrF++",
            ]
        ].rename(
            columns={
                "spBLEU": (
                    "sentence_oracle_spBLEU"
                ),
                "chrF++": (
                    "sentence_oracle_chrFpp"
                ),
            }
        ),
        on="country",
        how="left",
        validate="one_to_one",
    )
)

country_headroom_df[
    "sentence_oracle_spBLEU_gain"
] = (
    country_headroom_df[
        "sentence_oracle_spBLEU"
    ]
    - country_headroom_df[
        "current_spBLEU"
    ]
)

if (
    coordinate_oracle_country_scores
    is not None
):
    country_headroom_df = (
        country_headroom_df.merge(
            coordinate_oracle_country_scores[
                [
                    "country",
                    "spBLEU",
                    "chrF++",
                ]
            ].rename(
                columns={
                    "spBLEU": (
                        "coordinate_oracle_spBLEU"
                    ),
                    "chrF++": (
                        "coordinate_oracle_chrFpp"
                    ),
                }
            ),
            on="country",
            how="left",
            validate="one_to_one",
        )
    )

    country_headroom_df[
        "coordinate_oracle_spBLEU_gain"
    ] = (
        country_headroom_df[
            "coordinate_oracle_spBLEU"
        ]
        - country_headroom_df[
            "current_spBLEU"
        ]
    )

    country_headroom_df[
        "next_focus"
    ] = np.select(
        [
            country_headroom_df[
                "coordinate_oracle_spBLEU_gain"
            ].ge(3.0),
            country_headroom_df[
                "coordinate_oracle_spBLEU_gain"
            ].ge(1.5),
        ],
        [
            "routing_first",
            "hybrid_router_plus_new_expert",
        ],
        default=(
            "new_generation_expert_first"
        ),
    )

country_headroom_df.to_csv(
    ORACLE_DIR
    / "oracle_per_country_headroom.csv",
    index=False,
    encoding="utf-8-sig",
)

# ------------------------------------------------------------
# Largest qualitative opportunities
# ------------------------------------------------------------

if coordinate_oracle_df is not None:
    current_sentence_scores = (
        sentence_scores_df.merge(
            analysis_df[
                [
                    "source_id",
                    "selected_from_variant",
                ]
            ],
            left_on=[
                "source_id",
                "variant",
            ],
            right_on=[
                "source_id",
                "selected_from_variant",
            ],
            how="inner",
        )
        [
            [
                "source_id",
                "sentence_spBLEU",
                "sentence_chrFpp",
            ]
        ]
        .rename(
            columns={
                "sentence_spBLEU": (
                    "current_sentence_spBLEU"
                ),
                "sentence_chrFpp": (
                    "current_sentence_chrFpp"
                ),
            }
        )
    )

    coordinate_sentence_scores = (
        sentence_scores_df.merge(
            coordinate_oracle_df[
                [
                    "source_id",
                    "selected_variant",
                ]
            ],
            left_on=[
                "source_id",
                "variant",
            ],
            right_on=[
                "source_id",
                "selected_variant",
            ],
            how="inner",
        )
        [
            [
                "source_id",
                "sentence_spBLEU",
                "sentence_chrFpp",
            ]
        ]
        .rename(
            columns={
                "sentence_spBLEU": (
                    "oracle_sentence_spBLEU"
                ),
                "sentence_chrFpp": (
                    "oracle_sentence_chrFpp"
                ),
            }
        )
    )

    example_columns = BASE_COLUMNS.copy()

    for column in [
        "domain",
        "dialect",
        "speaker",
        "gender_direction",
    ]:
        if column in analysis_df.columns:
            example_columns.append(column)

    opportunity_df = (
        analysis_df[
            example_columns
            + [
                "current_prediction",
                "selected_from_variant",
            ]
        ]
        .merge(
            coordinate_oracle_df[
                [
                    "source_id",
                    "prediction",
                    "selected_variant",
                ]
            ].rename(
                columns={
                    "prediction": (
                        "oracle_prediction"
                    ),
                    "selected_variant": (
                        "oracle_selected_variant"
                    ),
                }
            ),
            on="source_id",
            how="left",
            validate="one_to_one",
        )
        .merge(
            current_sentence_scores,
            on="source_id",
            how="left",
        )
        .merge(
            coordinate_sentence_scores,
            on="source_id",
            how="left",
        )
    )

    opportunity_df[
        "sentence_chrFpp_delta"
    ] = (
        opportunity_df[
            "oracle_sentence_chrFpp"
        ]
        - opportunity_df[
            "current_sentence_chrFpp"
        ]
    )

    opportunity_df[
        "sentence_spBLEU_delta"
    ] = (
        opportunity_df[
            "oracle_sentence_spBLEU"
        ]
        - opportunity_df[
            "current_sentence_spBLEU"
        ]
    )

    opportunity_df = (
        opportunity_df.loc[
            opportunity_df[
                "oracle_prediction"
            ].ne(
                opportunity_df[
                    "current_prediction"
                ]
            )
        ]
        .sort_values(
            [
                "sentence_chrFpp_delta",
                "sentence_spBLEU_delta",
            ],
            ascending=False,
        )
        .reset_index(drop=True)
    )

    opportunity_df.head(500).to_csv(
        ORACLE_DIR
        / "largest_oracle_opportunities.csv",
        index=False,
        encoding="utf-8-sig",
    )

print("Final analysis ladder:")

display(
    oracle_summary_df[
        [
            "analysis_level",
            "system",
            "macro_spBLEU",
            "macro_chrFpp",
            "delta_vs_current_spBLEU",
            "honest_oof",
        ]
    ]
)

print(
    "\nCountry-level adapter/router priorities:"
)

display(
    country_headroom_df.sort_values(
        "current_spBLEU"
    )
)

if coordinate_oracle_summary is None:
    print(
        "\nRun Cell 10 before making "
        "the final decision."
    )

else:
    coordinate_gain = (
        coordinate_oracle_summary[
            "macro_spBLEU"
        ]
        - current_summary[
            "macro_spBLEU"
        ]
    )

    stable_sentence_gain = (
        sentence_chrf_oracle_summary[
            "macro_spBLEU"
        ]
        - current_summary[
            "macro_spBLEU"
        ]
    )

    oof_country_score = float(
        oof_router_summary_df.loc[
            oof_router_summary_df[
                "system"
            ].eq("oof_country"),
            "macro_spBLEU",
        ].iloc[0]
    )

    fine_oof_df = (
        oof_router_summary_df.loc[
            ~oof_router_summary_df[
                "system"
            ].isin([
                "oof_global",
                "oof_country",
            ])
        ]
    )

    best_fine_oof_score = (
        float(
            fine_oof_df[
                "macro_spBLEU"
            ].max()
        )
        if len(fine_oof_df)
        else oof_country_score
    )

    fine_oof_delta = (
        best_fine_oof_score
        - oof_country_score
    )

    print("\n" + "=" * 78)
    print(
        "COUNTRY ADAPTER / MoE DECISION"
    )
    print("=" * 78)

    print(
        "Coordinate candidate-pool headroom:",
        f"{coordinate_gain:+.4f}",
    )

    print(
        "Stable sentence-oracle headroom:",
        f"{stable_sentence_gain:+.4f}",
    )

    print(
        "Fine OOF routing vs OOF country:",
        f"{fine_oof_delta:+.4f}",
    )

    if coordinate_gain >= 3.0:
        print(
            "\nThe saved experts already contain "
            "enough latent score to cover the gap."
        )

        print(
            "Priority: train a deployable "
            "content-aware candidate router."
        )

    elif coordinate_gain >= 1.5:
        print(
            "\nThe current expert pool has useful "
            "but insufficient headroom."
        )

        print(
            "Priority: train r8 country adapters "
            "to add new experts, then route them."
        )

    else:
        print(
            "\nThe existing outputs cannot plausibly "
            "close the gap through routing alone."
        )

        print(
            "Priority: generation improvement and "
            "r8 country experts before MoE routing."
        )

    if fine_oof_delta > 0:
        print(
            "\nCountry/domain/turn metadata carries "
            "generalizable routing signal."
        )

        print(
            "A small metadata gate is justified."
        )

    else:
        print(
            "\nFine static metadata routing did not "
            "beat OOF country-only routing."
        )

        print(
            "Any future MoE gate should use source, "
            "context, and candidate-text features."
        )

if (
    coordinate_oracle_df is not None
    and len(opportunity_df)
):
    print(
        "\nLargest qualitative opportunities:"
    )

    display_columns = [
        "config",
        "source_text",
        "reference_arabic",
        "current_prediction",
        "selected_from_variant",
        "oracle_prediction",
        "oracle_selected_variant",
        "sentence_chrFpp_delta",
    ]

    if "domain" in opportunity_df.columns:
        display_columns.insert(
            1,
            "domain",
        )

    display(
        opportunity_df[
            display_columns
        ].head(20)
    )

print("\nSaved analysis outputs:")
print(ORACLE_DIR)

print(
    "\nWARNING: Oracle prediction files "
    "use DEV references and must not be submitted."
)

Final analysis ladder:


,analysis_level,system,macro_spBLEU,macro_chrFpp,delta_vs_current_spBLEU,honest_oof
0,corpus_coordinate_oracle,oracle_coordinate_corpus_spbleu,36.012678,49.378289,5.084675,False
1,sentence_oracle,oracle_sentence_spbleu,35.897330,49.403448,4.969327,False
2,sentence_oracle,oracle_sentence_chrfpp,35.527641,49.792009,4.599638,False
3,metadata_group_oracle,oracle_route_country_domain_turn,31.842600,46.202679,0.914597,False
4,metadata_group_oracle,oracle_route_country_domain,31.399517,45.896135,0.471514,False
5,metadata_group_oracle,oracle_route_country_turn,31.125025,45.704114,0.197022,False
6,current_system,92_mixed_best_checkpoint_variant_per_country,30.928003,45.594526,0.000000,False
7,metadata_group_oracle,oracle_route_country,30.928003,45.594526,0.000000,False
8,static_router_oof,oof_country,30.756904,45.492084,-0.171099,True
9,static_router_oof,oof_country_turn,30.697849,45.452443,-0.230153,True



Country-level adapter/router priorities:


,country,turns,current_spBLEU,current_chrFpp,sentence_oracle_spBLEU,sentence_oracle_chrFpp,sentence_oracle_spBLEU_gain,coordinate_oracle_spBLEU,coordinate_oracle_chrFpp,coordinate_oracle_spBLEU_gain,next_focus
4,MR,1114,17.380438,33.733013,21.393685,37.906977,4.013247,22.066587,37.309126,4.686148,routing_first
3,MA,1110,23.530143,39.376601,27.890393,43.678371,4.360250,28.489118,43.190678,4.958974,routing_first
10,YE,1118,27.380619,43.108824,31.930102,47.096973,4.549483,32.283325,46.727053,4.902706,routing_first
9,TN,1116,29.285225,43.452865,33.037994,47.697097,3.752769,33.772705,47.096758,4.487480,routing_first
2,LB,1118,31.701457,45.871215,36.517516,50.162828,4.816059,36.916171,49.775979,5.214714,routing_first
0,EG,1113,32.903189,46.892198,37.626217,51.208987,4.723028,38.322313,50.696390,5.419125,routing_first
7,SA,1110,33.166094,48.135685,37.757365,51.903948,4.591271,37.952262,51.616942,4.786167,routing_first
6,PS,1110,33.418402,47.762506,37.695395,51.632023,4.276993,38.020698,51.317832,4.602297,routing_first
1,JO,1113,35.301943,49.377895,40.623552,53.898451,5.321610,41.247103,53.497325,5.945160,routing_first
5,OM,1109,36.147408,49.843410,40.941611,53.920278,4.794204,41.147652,53.653689,5.000245,routing_first



COUNTRY ADAPTER / MoE DECISION
Coordinate candidate-pool headroom: +5.0847
Stable sentence-oracle headroom: +4.5996
Fine OOF routing vs OOF country: -0.0591

The saved experts already contain enough latent score to cover the gap.
Priority: train a deployable content-aware candidate router.

Fine static metadata routing did not beat OOF country-only routing.
Any future MoE gate should use source, context, and candidate-text features.

Largest qualitative opportunities:


,config,domain,source_text,reference_arabic,current_prediction,selected_from_variant,oracle_prediction,oracle_selected_variant,sentence_chrFpp_delta
0,EG,Commerce and transactions,What is it?,إيه هي؟,إيه الفكرة؟,05_retrieved_two_shot_with_participants,إيه هي؟,01_exact_training_parity,79.191752
1,PS,Energy and resources,And the cheaper ones?,والأرخص؟,طيب والرخصة؟,06_ckpt16500_retrieved_two_shot,والأرخص؟,07_ckpt16000_retrieved_two_shot,76.519471
2,EG,Logistics and transportation,And from Adly Mansour?,ومن عدلي منصور؟,ومن ادلي مانعور؟,05_retrieved_two_shot_with_participants,ومن عدلي منصور؟,00_previous_official_control,70.579300
3,LB,Healthcare and medical,"Everything has been fine, no complaints.",كل شي تمام، ما في مشاكل.,كل شي كان منيح، ما في أي شكوى.,07_ckpt16000_retrieved_two_shot,كل شي تمام، ما في مشاكل.,00_previous_official_control,67.614308
4,YE,Legal and financial,What are the chances they will listen?,ايش هي احتمالات انهم يسمعوا؟,كم هي فرص إنهم يسمعونا؟,06_ckpt16500_retrieved_two_shot,ايش هي احتمالات انهم يسمعون؟,07_ckpt16000_retrieved_two_shot,61.761928
5,OM,Professional and workplace,Do you have any plans for this weekend?,معك خطة حال اجازة نهاية الاسبوع؟,عندك اي خطط حال الويكند؟,05_retrieved_two_shot_with_participants,عندك اي خطط حال اجازة نهاية الاسبوع؟,06_ckpt16500_retrieved_two_shot,59.094775
6,SA,Legal and financial,What kind of conditions?,وش نوع الشروط؟,أي نوع من الشروط؟,06_ckpt16500_retrieved_two_shot,وش نوع الشروط؟,01_exact_training_parity,52.942885
7,MA,Commerce and transactions,And you as well. Goodbye.,وحتى انتي. بسلامة.,وانتي حتى نتي. مع السلامة.,05_retrieved_two_shot_with_participants,حتى انتي. بسلامة.,07_ckpt16000_retrieved_two_shot,51.879550
8,JO,Commerce and transactions,That sounds wonderful. Can you help us get a t...,حلو الكلام، بتقدر تساعدنا ناخد تكسي لهناك؟,ممتاز، بتقدر توصينا على تاكسي هناك؟,03_retrieved_two_shot,ممتاز، بتقدر تساعدنا ناخد تكسي لهناك؟,04_training_parity_with_participants,50.779706
9,PS,Everyday and social,"Oh, my love, safety upon you. Don't worry, I'l...",سلامتك يا حبيبتي .تقلقيش، رح اسويلك شوربة جاج ...,يييييييييييييييييييييييييييييييييييييييييييييي...,06_ckpt16500_retrieved_two_shot,يا حبيبتي، الله يحفظك. ما تقلقي، رح أعملك شورب...,04_training_parity_with_participants,49.982554



Saved analysis outputs:
/home/mabdallah/alexandriax_mt_14d/inference_variants/_oracle_analysis_v1

